Verify outcome encodings + mutation event counts (EPV check)

In [1]:
import pandas as pd, numpy as np, json

D = "/mnt/shared-workspace/shared/biomarker_val"
prim = pd.read_csv(f"{D}/cohort_ov_tcga_pan_can_atlas_2018.csv", low_memory=False)
print("PRIMARY cohort shape:", prim.shape)

# Outcome field encodings
for c in ["OS_STATUS","PFS_STATUS","NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT"]:
    if c in prim.columns:
        print(f"\n{c} value counts:")
        print(prim[c].value_counts(dropna=False).head(6))

# OS/PFS numeric ranges
for c in ["OS_MONTHS","PFS_MONTHS"]:
    if c in prim.columns:
        v = pd.to_numeric(prim[c], errors="coerce")
        print(f"\n{c}: n_valid={v.notna().sum()}  median={v.median():.1f}  range={v.min():.1f}-{v.max():.1f}")

# Mutation event counts per gene (EPV driver)
mut_cols = [c for c in prim.columns if c.startswith("mut_")]
print("\n=== Mutation event counts (PRIMARY n=585) ===")
counts = {c.replace("mut_",""): int(prim[c].sum()) for c in mut_cols}
for g,n in sorted(counts.items(), key=lambda x:-x[1]):
    print(f"  {g:10s} {n:4d}  ({100*n/len(prim):.1f}%)")
print(f"\nCCNE1 amplification (amp_CCNE1): {int(prim['amp_CCNE1'].sum())} ({100*prim['amp_CCNE1'].sum()/len(prim):.1f}%)")


PRIMARY cohort shape: (585, 57)

OS_STATUS value counts:
OS_STATUS
1:DECEASED    342
0:LIVING      232
NaN            11
Name: count, dtype: int64

PFS_STATUS value counts:
PFS_STATUS
1:PROGRESSION    408
0:CENSORED       167
NaN               10
Name: count, dtype: int64

NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT value counts:
NEW_TUMOR_EVENT_AFTER_INITIAL_TREATMENT
NaN    533
Yes     38
No      14
Name: count, dtype: int64

OS_MONTHS: n_valid=571  median=33.3  range=0.3-180.2

PFS_MONTHS: n_valid=571  median=14.7  range=0.3-180.2

=== Mutation event counts (PRIMARY n=585) ===
  PALB2         0  (0.0%)
  PTEN          0  (0.0%)
  KEAP1         0  (0.0%)
  CUL3          0  (0.0%)
  SLC31A1       0  (0.0%)
  TP53BP1       0  (0.0%)
  SHLD1         0  (0.0%)
  SHLD2         0  (0.0%)
  MAD2L2        0  (0.0%)
  MBD4          0  (0.0%)
  BRAF          0  (0.0%)
  KRAS          0  (0.0%)
  NRAS          0  (0.0%)
  CCNE1         0  (0.0%)
  BRCA1         0  (0.0%)
  BRCA2         0  (0.0%)
 

Sanity-gate mutation counts on primary cohort

In [3]:
import pandas as pd, json
d = "/mnt/shared-workspace/shared/biomarker_val"
df = pd.read_csv(f"{d}/cohort_ov_tcga_pan_can_atlas_2018.csv")
print("PRIMARY COHORT ov_tcga_pan_can_atlas_2018  shape:", df.shape)
mut_cols = [c for c in df.columns if c.startswith("mut_")]
print("\n=== Mutation frequencies (n=585) — SANITY CHECK ===")
for c in sorted(mut_cols):
    n = int(df[c].sum()); pct = 100*n/len(df)
    print(f"  {c:16s} {n:4d}  ({pct:5.1f}%)")
print(f"\n  amp_CCNE1        {int(df['amp_CCNE1'].sum()):4d}  ({100*df['amp_CCNE1'].sum()/len(df):5.1f}%)")

# Expected: TP53 ~65-96%, BRCA1/2 present, CCNE1 amp ~112 (19%)
tp53 = int(df["mut_TP53"].sum()); ccne1 = int(df["amp_CCNE1"].sum())
print("\n=== GATE ===")
print(f"  TP53 mutation {tp53}/585 = {100*tp53/585:.1f}%  -> {'PASS' if tp53>350 else 'FAIL'} (expect >60%)")
print(f"  CCNE1 amp {ccne1}/585 = {100*ccne1/585:.1f}%  -> {'PASS' if 90<=ccne1<=140 else 'CHECK'} (expect ~112)")
print(f"  BRCA1 present: {int(df['mut_BRCA1'].sum())}, BRCA2: {int(df['mut_BRCA2'].sum())}  -> {'PASS' if df['mut_BRCA1'].sum()>0 and df['mut_BRCA2'].sum()>0 else 'FAIL'}")

PRIMARY COHORT ov_tcga_pan_can_atlas_2018  shape: (585, 57)

=== Mutation frequencies (n=585) — SANITY CHECK ===
  mut_BRAF            2  (  0.3%)
  mut_BRCA1          18  (  3.1%)
  mut_BRCA2          15  (  2.6%)
  mut_CCNE1           2  (  0.3%)
  mut_CDK12          19  (  3.2%)
  mut_CUL3            3  (  0.5%)
  mut_KEAP1           2  (  0.3%)
  mut_KRAS            6  (  1.0%)
  mut_MAD2L2          0  (  0.0%)
  mut_MBD4            1  (  0.2%)
  mut_NF1            30  (  5.1%)
  mut_NRAS            5  (  0.9%)
  mut_PALB2           5  (  0.9%)
  mut_PTEN            7  (  1.2%)
  mut_RAD51C          0  (  0.0%)
  mut_RAD51D          0  (  0.0%)
  mut_SHLD1           0  (  0.0%)
  mut_SHLD2           4  (  0.7%)
  mut_SLC31A1         1  (  0.2%)
  mut_TP53          373  ( 63.8%)
  mut_TP53BP1         4  (  0.7%)

  amp_CCNE1         112  ( 19.1%)

=== GATE ===
  TP53 mutation 373/585 = 63.8%  -> PASS (expect >60%)
  CCNE1 amp 112/585 = 19.1%  -> PASS (expect ~112)
  BRCA1 present: 1

Parse outcomes and assess per-gene EPV feasibility

In [5]:
import pandas as pd, numpy as np
d = "/mnt/shared-workspace/shared/biomarker_val"
df = pd.read_csv(f"{d}/cohort_ov_tcga_pan_can_atlas_2018.csv")

# Parse outcome encodings (from summary: OS_STATUS "1:DECEASED", PFS_STATUS "1:PROGRESSION")
def parse_event(s):
    if pd.isna(s): return np.nan
    return 1 if str(s).startswith("1") else 0

df["os_event"] = df["OS_STATUS"].map(parse_event)
df["os_time"] = pd.to_numeric(df["OS_MONTHS"], errors="coerce")
df["pfs_event"] = df["PFS_STATUS"].map(parse_event)
df["pfs_time"] = pd.to_numeric(df["PFS_MONTHS"], errors="coerce")

print("=== Outcome parse check ===")
print("OS events:", int(df["os_event"].sum()), "/ valid", int(df["os_event"].notna().sum()))
print("PFS events:", int(df["pfs_event"].sum()), "/ valid", int(df["pfs_event"].notna().sum()))
print("OS_MONTHS median:", round(df["os_time"].median(),1), " PFS_MONTHS median:", round(df["pfs_time"].median(),1))

# EPV feasibility per gene: events-per-variable for a univariate Cox needs >=10 events IN the exposed group ideally
mut_cols = [c for c in df.columns if c.startswith("mut_")] + ["amp_CCNE1"]
os_ok = df["os_event"].notna()
print("\n=== EPV feasibility (min events across strata for OS) ===")
feas = []
for c in sorted(mut_cols):
    sub = df[os_ok]
    exposed = sub[sub[c]==1]
    n_exp = len(exposed); ev_exp = int(exposed["os_event"].sum())
    n_unexp = len(sub)-n_exp; ev_unexp = int(sub["os_event"].sum())-ev_exp
    min_ev = min(ev_exp, ev_unexp)
    tier = "TESTABLE" if (n_exp>=10 and ev_exp>=5) else ("BORDERLINE" if n_exp>=5 else "INSUFFICIENT_EVENTS")
    feas.append((c, n_exp, ev_exp, tier))
    print(f"  {c:16s} n_exposed={n_exp:4d} events_exposed={ev_exp:3d}  -> {tier}")

testable = [c for c,_,_,t in feas if t=="TESTABLE"]
print("\nTESTABLE genes (will get Cox HR):", testable)
df.to_csv(f"{d}/primary_cohort_parsed.csv", index=False)
print("\nSaved parsed primary cohort.")

=== Outcome parse check ===
OS events: 342 / valid 574
PFS events: 408 / valid 575
OS_MONTHS median: 33.3  PFS_MONTHS median: 14.7

=== EPV feasibility (min events across strata for OS) ===
  amp_CCNE1        n_exposed= 111 events_exposed= 79  -> TESTABLE
  mut_BRAF         n_exposed=   2 events_exposed=  1  -> INSUFFICIENT_EVENTS
  mut_BRCA1        n_exposed=  18 events_exposed=  4  -> BORDERLINE
  mut_BRCA2        n_exposed=  15 events_exposed=  7  -> TESTABLE
  mut_CCNE1        n_exposed=   2 events_exposed=  2  -> INSUFFICIENT_EVENTS
  mut_CDK12        n_exposed=  19 events_exposed= 13  -> TESTABLE
  mut_CUL3         n_exposed=   3 events_exposed=  2  -> INSUFFICIENT_EVENTS
  mut_KEAP1        n_exposed=   2 events_exposed=  2  -> INSUFFICIENT_EVENTS
  mut_KRAS         n_exposed=   6 events_exposed=  4  -> BORDERLINE
  mut_MAD2L2       n_exposed=   0 events_exposed=  0  -> INSUFFICIENT_EVENTS
  mut_MBD4         n_exposed=   1 events_exposed=  1  -> INSUFFICIENT_EVENTS
  mut_NF1     

Detail HR/CI/p and flag significant associations

In [7]:
import json
D = "/mnt/shared-workspace/shared/biomarker_val"
r = json.load(open(f"{D}/validation_52_results.json"))
print(f"Cohort: {r['cohort']} n={r['n']}\n")
print(f"{'gene':14s} {'tier':22s} {'OS HR (95% CI) p':32s} {'PFS HR (95% CI) p':32s} PH")
print("-"*115)
for g, e in r["genes"].items():
    def fmt(x):
        if x.get("HR") is None: return f"{'—':32s}"
        ph = "!" if x.get("PH_violated") else " "
        return f"{x['HR']:.2f} ({x['CI_low']:.2f}-{x['CI_high']:.2f}) p={x['p']:<6}".ljust(32)
    os_s, pfs_s = fmt(e["OS"]), fmt(e["PFS"])
    phos = e["OS"].get("PH_p"); 
    print(f"{g:14s} {e['gene_tier']:22s} {os_s} {pfs_s} PHos={phos}")

# Highlight statistically significant (CI excludes 1)
print("\n=== Statistically significant associations (95% CI excludes 1.0) ===")
for g, e in r["genes"].items():
    for ep in ("OS","PFS"):
        x = e[ep]
        if x.get("HR") and (x["CI_low"]>1 or x["CI_high"]<1):
            direction = "WORSE (resistance/poor prognosis)" if x["HR"]>1 else "BETTER (sensitizing/good prognosis)"
            print(f"  {g} [{ep}]: HR={x['HR']} ({x['CI_low']}-{x['CI_high']}) p={x['p']} -> {direction}")

Cohort: ov_tcga_pan_can_atlas_2018 n=585

gene           tier                   OS HR (95% CI) p                 PFS HR (95% CI) p                PH
-------------------------------------------------------------------------------------------------------------------
amp_CCNE1      COHORT_VALIDATED_RUO   1.51 (1.17-1.94) p=0.0015        1.36 (1.07-1.73) p=0.0119        PHos=0.4788
mut_BRAF       INSUFFICIENT_EVENTS    —                                —                                PHos=None
mut_BRCA1      COHORT_VALIDATED_RUO   0.57 (0.21-1.53) p=0.2627        0.79 (0.41-1.53) p=0.4897        PHos=0.5378
mut_BRCA2      COHORT_VALIDATED_RUO   0.42 (0.20-0.88) p=0.0219        0.36 (0.17-0.75) p=0.0069        PHos=0.0866
mut_CCNE1      INSUFFICIENT_EVENTS    —                                —                                PHos=None
mut_CDK12      COHORT_VALIDATED_RUO   1.14 (0.66-1.99) p=0.6372        1.16 (0.68-1.97) p=0.5951        PHos=0.7618
mut_CUL3       INSUFFICIENT_EVENTS    —    

Locate v2 edit content in worker-0 notebook

In [9]:
import json, re

nb_path = "/mnt/results/execution_trace/worker-0.ipynb"
with open(nb_path) as f:
    nb = json.load(f)

# Find cells that contain v2 edit content — look for the key markers
markers = ["_derive_ccne1_amp_from_cna", "detect_serial_sae_divergence", "WIRED_PROGNOSTIC_ONLY_V1",
           "SERIAL_SAE_STRUCTURAL", "_add_profile_aliases", "ccne1_cna_status",
           "PILOT_DEMONSTRATION_N4", "DISCOVERY_ONLY_NOT_CALIBRATED", "ctdna_tier",
           "test_res_ov_p0_006", "test_res_ov_p0_009", "test_harness_e_ctdna"]

hits = []
for i, cell in enumerate(nb.get("cells", [])):
    src = "".join(cell.get("source", []))
    matched = [m for m in markers if m in src]
    if matched:
        hits.append((i, cell.get("cell_type"), len(src), matched))

print(f"Total cells: {len(nb['cells'])}")
print(f"Cells with v2 markers: {len(hits)}\n")
for i, ct, ln, matched in hits:
    print(f"cell {i:3d} [{ct}] {ln:6d} chars | markers: {matched}")


Total cells: 1521
Cells with v2 markers: 8

cell 1354 [code]   3025 chars | markers: ['WIRED_PROGNOSTIC_ONLY_V1']
cell 1368 [code]   7770 chars | markers: ['WIRED_PROGNOSTIC_ONLY_V1']
cell 1372 [code]   2410 chars | markers: ['WIRED_PROGNOSTIC_ONLY_V1', 'ctdna_tier']
cell 1376 [code]   2277 chars | markers: ['WIRED_PROGNOSTIC_ONLY_V1', 'ctdna_tier']
cell 1382 [code]   3408 chars | markers: ['DISCOVERY_ONLY_NOT_CALIBRATED']
cell 1482 [code]   8552 chars | markers: ['WIRED_PROGNOSTIC_ONLY_V1']
cell 1500 [code]   4317 chars | markers: ['WIRED_PROGNOSTIC_ONLY_V1']
cell 1512 [code]   3936 chars | markers: ['WIRED_PROGNOSTIC_ONLY_V1']


Inspect largest v2 notebook cells

In [11]:
import json
with open("/mnt/results/execution_trace/worker-0.ipynb") as f:
    nb = json.load(f)

for idx in [1368, 1482]:
    src = "".join(nb["cells"][idx].get("source", []))
    print("="*80)
    print(f"CELL {idx} ({len(src)} chars)")
    print("="*80)
    print(src[:3500])
    print("..." if len(src) > 3500 else "")
    print()


CELL 1368 (7770 chars)
import json, datetime

def overall_tier(gene, r):
    """Best honest earned tier across somatic + expr channels."""
    order = {"EXTERNALLY_VALIDATED_RUO":6,"COHORT_VALIDATED_RUO":5,"CALIBRATED_RUO":4,
             "BORDERLINE_RUO":3,"COHORT_TESTED_NULL":2,"LITERATURE_ONLY":1,
             "INSUFFICIENT_EVENTS":0,"NOT_ON_ARRAY":0,"NOT_TESTED":0}
    cands = [r["somatic_tier"], r["expr_tier"]]
    best = max(cands, key=lambda t: order.get(t,0))
    # If best is a 'validated' but the OTHER channel is only insufficient/null, that's fine — best honest.
    # If NOTHING reached a real association, and it started literature-only, keep LITERATURE_ONLY as floor
    if order.get(best,0) <= 2:
        # tested-null or insufficient -> the register said LITERATURE_ONLY originally; honest label is
        # "tested on real cohort, no OV-specific association" which is STRONGER info than literature-only
        if best in ("INSUFFICIENT_EVENTS","NOT_ON_ARRAY","NOT_TESTED"):
  

Extract v2 code references from transcript

In [13]:
import json

TX = "/mnt/results/execution_trace/transcript.jsonl"

# Pull transcript lines that reference each v2 target and capture the surrounding tool-call payloads.
targets = {
    "detect_serial_sae_divergence": [],
    "_derive_ccne1_amp_from_cna": [],
    "_add_profile_aliases": [],
    "SERIAL_SAE_STRUCTURAL": [],
}

records = []
with open(TX) as f:
    for lineno, line in enumerate(f):
        for t in targets:
            if t in line:
                targets[t].append(lineno)
        # capture the raw record for later extraction if it's a tool call with file content
        if any(t in line for t in targets):
            try:
                obj = json.loads(line)
                records.append((lineno, obj))
            except Exception:
                records.append((lineno, {"_raw_len": len(line)}))

for t, locs in targets.items():
    print(f"{t}: lines {locs}")
print(f"\nCaptured {len(records)} candidate records")
# Inspect the structure of one record to understand transcript schema
if records:
    ln, obj = records[0]
    print(f"\n--- record at line {ln}, keys: {list(obj.keys()) if isinstance(obj,dict) else type(obj)} ---")
    print(json.dumps(obj, indent=2)[:1500] if isinstance(obj, dict) else str(obj)[:1500])


detect_serial_sae_divergence: lines [44365, 44372, 44374, 44376, 44382, 44411, 44412, 44415, 44741, 44743, 44787, 44788, 44800, 44801]
_derive_ccne1_amp_from_cna: lines [44328, 44332, 44344, 44512, 44730, 44741, 44743, 44787, 44788, 44800, 44801]
_add_profile_aliases: lines [16172, 44275, 44318, 44319, 44320, 44321, 44322, 44324, 44326, 44787, 44788, 44800, 44801]
SERIAL_SAE_STRUCTURAL: lines [44368, 44372, 44411, 44415, 44671, 44787, 44788]

Captured 30 candidate records

--- record at line 16172, keys: ['i', 'type', 'content', 'tool_name', 'content_chunk'] ---
{
  "i": 2558,
  "type": "tool",
  "content": "eys\"\n      - \"tests/audit/test_therapy_fit_v2_slop_proofs.py::test_closure_tf2_slop_002_missing_expression_exposes_locked_gate_status\"\n    proof_artifact:\n      test: \"tests/audit/test_therapy_fit_v2_slop_proofs.py::test_slop_02_spe_shape_mismatch_risk_is_real_expr_none_gate_locked\"\n    owner: backend\n    remediation_type: shape_fix + response_honesty_label\n    validatio

Reassemble transcript region A with v2 edits

In [15]:
import json

TX = "/mnt/results/execution_trace/transcript.jsonl"
lines = open(TX).read().splitlines()

# Reassemble the two dense clusters: 44270-44420 and 44720-44810
def dump_region(lo, hi, label):
    print("="*90); print(f"REGION {label}: lines {lo}-{hi}"); print("="*90)
    for ln in range(lo, min(hi, len(lines))):
        try:
            obj = json.loads(lines[ln])
        except Exception:
            continue
        tn = obj.get("tool_name","")
        typ = obj.get("type","")
        content = obj.get("content","")
        # Only show tool/assistant records with substantive content
        if content and (typ in ("tool","assistant","user")):
            cc = obj.get("content_chunk",{})
            chunk_info = f"[chunk {cc.get('index','')}/{cc.get('count','')}]" if cc else ""
            print(f"\n--- L{ln} type={typ} tool={tn} {chunk_info} ---")
            print(content[:1400])

dump_region(44315, 44420, "A (builder/ayesha/ovarian edits)")


REGION A (builder/ayesha/ovarian edits): lines 44315-44420

--- L44315 type=tool tool=ExecuteCode [chunk 2/2] ---
4110 (baseline 0.98) chr19 5Mb-window median CN:
  KEAP1@~10.6Mb window: 1.23 | CCNE1@~30.3Mb window: 0.95
  amplified windows (>=1.5x baseline): none
  fraction of chr19 windows amplified: 0.00 -> FOCAL


--- L44316 type=assistant tool= [chunk 1/2] ---
This scientific check is exactly why I don't rubber-stamp. Important nuances the raw calls hid:

- **JBLAB-4225 CCNE1 amp is genuinely FOCAL** (only the two 5Mb windows spanning 19q12 at 25–30 Mb are elevated; 17% of chr19) — this is a clean, real focal CCNE1 amplification.
- **IM_373 is BROAD** — the whole 19q (and part of 19p) is elevated, with CCNE1 sitting on a peak (3.51 within a broadly-gained arm). So CCNE1 is amplified, but it's part of a large-scale chr19 gain, not an isolated focal spike. Still a real CCNE1 amplification — but I'll label it `AMP_BROAD` vs `AMP_FOCAL` honestly.
- **The KEAP1 "amplifications" are the

Reassemble ovarian serial-SAE edits from transcript

In [17]:
import json
TX = "/mnt/results/execution_trace/transcript.jsonl"
lines = open(TX).read().splitlines()

def dump_region(lo, hi, label, maxchars=1500):
    print("="*90); print(f"REGION {label}: lines {lo}-{hi}"); print("="*90)
    for ln in range(lo, min(hi, len(lines))):
        try:
            obj = json.loads(lines[ln])
        except Exception:
            continue
        tn = obj.get("tool_name",""); typ = obj.get("type",""); content = obj.get("content","")
        if content and (typ in ("tool","assistant")):
            cc = obj.get("content_chunk",{})
            ci = f"[chunk {cc.get('index','')}/{cc.get('count','')}]" if cc else ""
            print(f"\n--- L{ln} {typ} {tn} {ci} ---")
            print(content[:maxchars])

# The serial-SAE + ovarian.py + schemas.py + test files cluster
dump_region(44360, 44420, "B (ovarian serial-SAE + schema)")


REGION B (ovarian serial-SAE + schema): lines 44360-44420

--- L44360 tool Bash [chunk 3/3] ---
ec = baseline_sae.get("mechanism_vector", [])
246:    # in SAE service handles the vector properly. Here we will use a simplified assumption
262:    is_pop_baseline = baseline_sae.get("is_population_average", False)
324:async def detect_genomic_resistance(
411:async def detect_ca125_kinetics(
546:async def detect_diamond_features(
547:    current_sae: Dict
550:    Detect "Diamond" True SAE Features (Verified Resistance Drivers).
553:    Source: SAE_FAILURE_POSTMORTEM.mdc
555:    logger.info("Detecting DIAMOND_SAE_FEATURES...")
557:    # We expect `current_sae` to potentially have a 'features' or 'activations' dict 
562:    features_map = current_sae.get("features", current_sae) # Try nested, then self


--- L44361 assistant   ---
There's already a rich SAE signal framework (`detect_restoration`, `detect_escape`, `detect_genomic_resistance`, `detect_diamond_features`) that consumes `current_s

Reassemble register + production readiness + final test regions

In [19]:
import json
TX = "/mnt/results/execution_trace/transcript.jsonl"
lines = open(TX).read().splitlines()

# Region with the ISSUE_REGISTER edits, production readiness Section 11, and final test verification
def dump_region(lo, hi, label, maxchars=1600):
    out = []
    for ln in range(lo, min(hi, len(lines))):
        try:
            obj = json.loads(lines[ln])
        except Exception:
            continue
        tn = obj.get("tool_name",""); typ = obj.get("type",""); content = obj.get("content","")
        if content and (typ in ("tool","assistant")):
            cc = obj.get("content_chunk",{})
            ci = f"[{cc.get('index','')}/{cc.get('count','')}]" if cc else ""
            out.append(f"\n--- L{ln} {typ} {tn} {ci} ---\n{content[:maxchars]}")
    print("="*90); print(f"REGION {label}: {lo}-{hi}"); print("="*90)
    print("".join(out))

# Final verification + register + section 11 cluster
dump_region(44720, 44810, "C (register/prod-readiness/final tests)")


REGION C (register/prod-readiness/final tests): 44720-44810

--- L44720 assistant   ---
v3 base and w3 artifacts both scan clean — zero password hits. Now I must reconstruct the lost v2 work. Let me inform the user of the OOM event honestly, then re-clone and rebuild. This is material — the user needs to know their `/workspace` was wiped mid-task.
--- L44721 assistant   ---
Heads-up: the sandbox hit an out-of-memory restart during the final secret-scan and wiped `/workspace`. Impact is contained:
- **Preserved** (S3-backed): all v3 edits (`service.py`, `test_brm_bundle.py`, `BLOCKER_REGISTER.yaml` — v3 T-series already at 10/10), and every w3 data artifact (all scan clean, zero password hits).
- **Lost**: the v2 repo working tree with my staged (uncommitted) edits — because you chose "review before commit," they were never pushed. I have the exact spec for every v2 edit and the data that drove them, so I'm re-cloning `crispro-backend-v2` and rebuilding now. No data or analysis is lost,

Locate full bodies of new v2 test files in transcript

In [21]:
import json
TX = "/mnt/results/execution_trace/transcript.jsonl"
lines = open(TX).read().splitlines()

# Find the Write tool calls for the two new test files and reassemble their full content.
# Write payloads appear as assistant tool_use with the file content. In this transcript schema,
# the content is chunked. Let me search for the test file names and grab surrounding write content.
def find_write_content(fname_frag):
    """Collect all content chunks that belong to the Write of a given file."""
    collected = []
    for ln, line in enumerate(lines):
        if fname_frag in line:
            try:
                obj = json.loads(line)
            except Exception:
                continue
            # Look for chunks near this that contain python code (def test_, import, assert)
            content = obj.get("content","")
            if any(k in content for k in ["def test_", "import ", "assert ", "@pytest"]):
                collected.append((ln, obj.get("type"), obj.get("tool_name"), content))
    return collected

for frag in ["test_ccne1_cna_integration_pilot", "test_serial_sae_signal_pilot"]:
    print("="*80); print(f"SEARCH: {frag}"); print("="*80)
    hits = find_write_content(frag)
    print(f"{len(hits)} content-bearing records")
    for ln, typ, tn, content in hits[:6]:
        print(f"\n--- L{ln} {typ} {tn} ---")
        print(content[:1200])


SEARCH: test_ccne1_cna_integration_pilot
3 content-bearing records

--- L44516 tool Bash ---
=== stash ONLY the 2 source files that touch therapy_fit/builder ===
Saved working directory and index state WIP on feature/biomarker-subsystem: 0169ca4 feat(resistance): v3 spine — survival-outcome channel, Track B calibrated classifier, ctDNA prognostic-only, PLCO outcome-context, in-silico analytical validation (RUO)
=== full audit dir with my source edits REMOVED (test edits + register + ovarian/schemas still present) ===
    from api.routers.ayesha_therapy_fit import _derive_ccne1_amp_from_cna
E   ImportError: cannot import name '_derive_ccne1_amp_from_cna' from 'api.routers.ayesha_therapy_fit' (/workspace/crispro-backend-v2/api/routers/ayesha_therapy_fit.py)
=========================== short test summary info ============================
ERROR tests/audit/test_ccne1_cna_integration_pilot.py
!!!!!!!!!!!!!!!!!!!! Interrupted: 1 error during collection !!!!!!!!!!!!!!!!!!!!
1 error in 2.39s
=

Pull register entry content for 006/009 from transcript

In [23]:
import json
TX = "/mnt/results/execution_trace/transcript.jsonl"
lines = open(TX).read().splitlines()

# Pull the register entries region (44394+) for RES-OV-P0-006 / 009
def region_text(lo, hi, only_tool=None):
    out=[]
    for ln in range(lo, min(hi,len(lines))):
        try: obj=json.loads(lines[ln])
        except: continue
        c=obj.get("content","");tn=obj.get("tool_name","");typ=obj.get("type","")
        if c and typ in ("tool","assistant"):
            if only_tool and tn!=only_tool: continue
            out.append((ln,typ,tn,c))
    return out

print("="*80,"\nREGISTER ENTRIES (006/009/SLOP-004) region 44394-44520\n","="*80)
for ln,typ,tn,c in region_text(44394,44470):
    print(f"\n--- L{ln} {typ} {tn} ---\n{c[:1400]}")


REGISTER ENTRIES (006/009/SLOP-004) region 44394-44520

--- L44394 tool Read ---
Lines 460-594 (use offset=595 to see more)

  - issue_id: RES-OV-P0-006
    title: "ctDNA accepted by Kill Chain but NOT wired into Prophet resistance probability"
    severity: P1
    deception_risk: scientific
    exact_location: "api/services/resistance_prophet_service.py::predict_resistance — no ctDNA parameter; api/services/kill_chain/policy.py::add_ctdna_signal — accepts ctDNA"
    closure_criteria:
      - "Prophet provenance explicitly documents ctdna_status='NOT_WIRED_INTO_PROPHET'"
      - "ctdna_note explains the gap to consumers"
      - "Proof test passes: test_res_ov_p0_006_ctdna_stub_in_prophet_provenance"
    patch_targets:
      - "api/services/resistance_prophet_service.py::ClinicalHeuristicEngine.run + predict_resistance serialized path — add ctdna_status to provenance"
    validation_tests:
      - "tests/audit/test_resistance_prophet_production.py::test_res_ov_p0_006_ctdna_stub_in_prop

Generate ranked-signal and breakpoint-architecture figures

In [25]:
import matplotlib
matplotlib.rcParams['font.family'] = ['Liberation Sans', 'Arimo', 'DejaVu Sans']
matplotlib.rcParams['svg.fonttype'] = 'none'
import matplotlib.pyplot as plt
import pandas as pd, numpy as np

DST = "/mnt/results/biomarker_subsystem/real_data_validation"
# Phylo palette
BLACK="#000000"; GREEN="#75A025"; ORANGE="#FF9400"; BLUE="#0279EE"; YELLOW="#E9ED4C"; PINK="#FD9BED"

# ---- Panel A: ranked external C-index (prognostic signals with a C-index) ----
rt = pd.read_csv(f"{DST}/ranked_signal_table.csv")
prog = rt[rt["best_ext_cindex"].notna()].copy().sort_values("best_ext_cindex")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))

ax = axes[0]
ax.barh(prog["signal"], prog["best_ext_cindex"], color=[GREEN if r=="REPLICATES_1of2" else ORANGE for r in prog["replication"]], edgecolor=BLACK, height=0.5)
ax.axvline(0.5, color=BLACK, ls="--", lw=1.2, label="null (C=0.5)")
for y,(c,coh) in enumerate(zip(prog["best_ext_cindex"], prog["best_ext_cindex_cohort"])):
    ax.text(c+0.006, y, f"{c:.3f}\n({coh})", va="center", fontsize=8)
ax.set_xlim(0.45, 0.72)
ax.set_xlabel("Best independent external C-index")
ax.set_title("A. Prognostic signals — best external discrimination\n(all discovery_only; none pass the >=2-cohort gate)", fontsize=10, loc="left")
ax.legend(fontsize=8, loc="lower right")
ax.spines[["top","right"]].set_visible(False)

# ---- Panel B: breakpoint delta by subject, architecture-coded ----
bp = pd.read_csv(f"{DST}/breakpoint_architecture_pilot.csv").sort_values("breakpoint_delta")
colors = {"dispersed_stochastic_CIN":ORANGE, "intermediate_rearrangement":YELLOW,
          "modest_gain":GREEN, "chromothripsis_like_focal_catastrophe":PINK}
ax = axes[1]
bars = ax.barh([f"Subj {s}" for s in bp["subject"]], bp["breakpoint_delta"],
               color=[colors.get(a, BLUE) for a in bp["architecture_signature"]], edgecolor=BLACK, height=0.5)
ax.axvline(50, color=BLACK, ls="--", lw=1.2, label="high-breakpoint threshold (+50)")
ax.axvline(0, color=BLACK, lw=0.8)
for y,(d,a,nc) in enumerate(zip(bp["breakpoint_delta"], bp["architecture_signature"], bp["n_chromosomes_gained"])):
    ax.text(d + (6 if d>=0 else -6), y, f"{d:+d}\n{a.replace('_',' ')}\n({nc} chr)", va="center",
            ha="left" if d>=0 else "right", fontsize=7.5)
ax.set_xlim(-40, 320)
ax.set_xlabel("Breakpoint delta (relapse - diagnosis)")
ax.set_title("B. Structural evolution (dx->relapse) — 4 processed subjects\nreal QDNAseq sWGS; Subj260 pending download", fontsize=10, loc="left")
ax.legend(fontsize=8, loc="lower right")
ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig(f"{DST}/fig_ranked_signals_and_breakpoints.png", dpi=150, bbox_inches="tight")
plt.savefig(f"{DST}/fig_ranked_signals_and_breakpoints.svg", bbox_inches="tight")
plt.close()
print("saved fig_ranked_signals_and_breakpoints.png/.svg")
print("\nBreakpoint architecture summary:")
print(bp[["subject","breakpoint_delta","architecture_signature","n_chromosomes_gained","top_focus_fraction"]].to_string(index=False))


saved fig_ranked_signals_and_breakpoints.png/.svg

Breakpoint architecture summary:
 subject  breakpoint_delta     architecture_signature  n_chromosomes_gained  top_focus_fraction
     109                -7                modest_gain                     3               0.571
     119                18 intermediate_rearrangement                    11               0.250
       6               101   dispersed_stochastic_CIN                    20               0.119
     191               253   dispersed_stochastic_CIN                    21               0.115


Regenerate figure with fixed annotation placement in Panel B

In [27]:
import matplotlib.pyplot as plt
import pandas as pd

DST = "/mnt/results/biomarker_subsystem/real_data_validation"
BLACK="#000000"; GREEN="#75A025"; ORANGE="#FF9400"; BLUE="#0279EE"; YELLOW="#E9ED4C"; PINK="#FD9BED"

rt = pd.read_csv(f"{DST}/ranked_signal_table.csv")
prog = rt[rt["best_ext_cindex"].notna()].copy().sort_values("best_ext_cindex")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

# Panel A (unchanged, was clean)
ax = axes[0]
ax.barh(prog["signal"], prog["best_ext_cindex"], color=[GREEN if r=="REPLICATES_1of2" else ORANGE for r in prog["replication"]], edgecolor=BLACK, height=0.5)
ax.axvline(0.5, color=BLACK, ls="--", lw=1.2, label="null (C=0.5)")
for y,(c,coh) in enumerate(zip(prog["best_ext_cindex"], prog["best_ext_cindex_cohort"])):
    ax.text(c+0.006, y, f"{c:.3f} ({coh})", va="center", fontsize=8)
ax.set_xlim(0.45, 0.75)
ax.set_xlabel("Best independent external C-index")
ax.set_title("A. Prognostic signals — best external discrimination\n(all discovery_only; none pass the >=2-cohort gate)", fontsize=10, loc="left")
ax.legend(fontsize=8, loc="lower right")
ax.spines[["top","right"]].set_visible(False)

# Panel B — fixed: annotations at a FIXED x well clear of the axis, left-aligned
bp = pd.read_csv(f"{DST}/breakpoint_architecture_pilot.csv").sort_values("breakpoint_delta")
colors = {"dispersed_stochastic_CIN":ORANGE, "intermediate_rearrangement":YELLOW,
          "modest_gain":GREEN, "chromothripsis_like_focal_catastrophe":PINK}
ax = axes[1]
ax.barh([f"Subj {s}" for s in bp["subject"]], bp["breakpoint_delta"],
        color=[colors.get(a, BLUE) for a in bp["architecture_signature"]], edgecolor=BLACK, height=0.5)
ax.axvline(50, color=BLACK, ls="--", lw=1.2, label="high-breakpoint threshold (+50)")
ax.axvline(0, color=BLACK, lw=0.8)
ANNOT_X = 120  # fixed annotation start, clear of small/negative bars and y-axis labels
for y,(d,a,nc) in enumerate(zip(bp["breakpoint_delta"], bp["architecture_signature"], bp["n_chromosomes_gained"])):
    ax.text(ANNOT_X, y, f"{d:+d}  |  {a.replace('_',' ')}  ({nc} chr)", va="center",
            ha="left", fontsize=8.5, color=BLACK)
ax.set_xlim(-40, 340)
ax.set_xlabel("Breakpoint delta (relapse - diagnosis)")
ax.set_title("B. Structural evolution (dx->relapse) — 4 processed subjects\nreal QDNAseq sWGS; Subj 260 pending download", fontsize=10, loc="left")
ax.legend(fontsize=8, loc="upper right")
ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
plt.savefig(f"{DST}/fig_ranked_signals_and_breakpoints.png", dpi=150, bbox_inches="tight")
plt.savefig(f"{DST}/fig_ranked_signals_and_breakpoints.svg", bbox_inches="tight")
plt.close()
print("regenerated with fixed annotation placement")


regenerated with fixed annotation placement


Update Subject 260 ledger row with real computed CN values

In [29]:
import pandas as pd, json, shutil, os

W = "/mnt/shared-workspace/shared/data_acq/w3"
A = f"{W}/subj260_analysis"
led_path = f"{W}/ledger/subject_ledger.csv"

# --- 1. Update Subject 260 row in the master ledger with REAL computed values ---
led = pd.read_csv(led_path, dtype=str)
meta = json.load(open(f"{A}/subj260_analysis_meta.json"))
arch = meta["architecture"]

mask = led["subject"] == "260"
assert mask.sum() == 1, f"expected 1 Subject-260 row, got {mask.sum()}"
led.loc[mask, "processed"] = "True"
led.loc[mask, "dx_breakpoints"] = str(arch["dx_breakpoints"])
led.loc[mask, "relapse_breakpoints"] = str(arch["relapse_breakpoints"])
led.loc[mask, "breakpoint_delta"] = str(arch["breakpoint_delta"])
led.loc[mask, "architecture_signature"] = arch["architecture_signature"]
led.loc[mask, "n_chromosomes_gained"] = str(arch["n_chromosomes_gained"])
led.loc[mask, "cn_status"] = "COMPUTED_QDNAseq_hg19_30kb"
# high_breakpoint flag: consistent with pilot definition (relapse_breakpoints >= 100 OR net_gained >= 50)
hb = (arch["relapse_breakpoints"] >= 100) or (arch["net_gained_breakpoints"] >= 50)
led.loc[mask, "high_breakpoint"] = str(hb)
led.to_csv(led_path, index=False)
print("=== Updated Subject 260 ledger row ===")
print(led.loc[mask].T.to_string())

# --- 2. Recompute the high-breakpoint ledger summary (all processed subjects) ---
proc = led[led["processed"] == "True"].copy()
for c in ["dx_breakpoints","relapse_breakpoints","breakpoint_delta","n_chromosomes_gained"]:
    proc[c] = pd.to_numeric(proc[c], errors="coerce")
proc = proc.sort_values("breakpoint_delta", ascending=False)
cols = ["subject","dx_sample","relapse_sample","histology","dx_breakpoints","relapse_breakpoints",
        "breakpoint_delta","architecture_signature","n_chromosomes_gained","high_breakpoint","cn_status"]
cols = [c for c in cols if c in proc.columns]
print("\n=== All processed subjects (ranked by breakpoint_delta) ===")
print(proc[cols].to_string(index=False))
print(f"\nTotal subjects in ledger: {len(led)} | processed (real CN): {len(proc)} | high_breakpoint: {(proc['high_breakpoint']=='True').sum()}")


=== Updated Subject 260 ledger row ===
                                               169
subject                                        260
dx_sample                                   IM_426
dx_egaf                            EGAF00008095344
dx_egan                            EGAN00004327464
dx_site                                      Ovary
relapse_sample                         JBLAB-19323
relapse_egaf                       EGAF00008095415
relapse_egan                       EGAN00004327572
relapse_site                               Omentum
histology                                   female
processed                                     True
dx_breakpoints                                  75
relapse_breakpoints                            135
breakpoint_delta                                60
architecture_signature    dispersed_stochastic_CIN
n_chromosomes_gained                            16
cn_status               COMPUTED_QDNAseq_hg19_30kb
high_breakpoint                            

Regenerate breakpoint figure including Subject 260

In [31]:
import matplotlib
matplotlib.rcParams['font.family'] = ['Liberation Sans', 'Arimo', 'DejaVu Sans']
matplotlib.rcParams['svg.fonttype'] = 'none'
import matplotlib.pyplot as plt
import pandas as pd, numpy as np

W = "/mnt/shared-workspace/shared/data_acq/w3"
R = "/mnt/results/biomarker_subsystem/real_data_validation"

# --- load data ---
arch = pd.read_csv(f"{W}/subj260_analysis/breakpoint_architecture_with_subj260.csv")
ranked = pd.read_csv(f"{W}/report_assets/ranked_signal_table.csv")

# Phylo palette
PHY = dict(black="#000000", cream="#ECE9E2", ivory="#FAF9F3", yellow="#E9ED4C",
           orange="#FF9400", green="#75A025", pink="#FD9BED", blue="#0279EE")
sig_color = {
    "dispersed_stochastic_CIN": PHY["blue"],
    "intermediate_rearrangement": PHY["orange"],
    "modest_gain": PHY["green"],
    "chromothripsis_like_focal_catastrophe": PHY["pink"],
}

fig, axes = plt.subplots(1, 3, figsize=(17, 5.4))
fig.subplots_adjust(wspace=0.42, left=0.055, right=0.985, top=0.86, bottom=0.16)

# ---------- Panel A: ranked signals (best external C-index) ----------
ax = axes[0]
r = ranked.copy()
r["label"] = r["rank"].astype(str) + ". " + r["signal"].str.replace(r"\s*\(.*\)","",regex=True).str[:26]
r["cidx_plot"] = pd.to_numeric(r["best_ext_cindex"], errors="coerce")
# consensusOV has no C-index (descriptive) -> mark separately
ypos = np.arange(len(r))[::-1]
colors = []
for _, row in r.iterrows():
    if pd.isna(row["cidx_plot"]):
        colors.append(PHY["cream"])
    elif row["replication"] == "REPLICATES_1of2":
        colors.append(PHY["orange"])
    else:
        colors.append(PHY["green"])
vals = r["cidx_plot"].fillna(0.5)
bars = ax.barh(ypos, vals, color=colors, edgecolor=PHY["black"], height=0.6)
ax.axvline(0.5, color=PHY["black"], ls=":", lw=1.2, alpha=0.7)
ax.set_yticks(ypos); ax.set_yticklabels(r["label"], fontsize=9)
ax.set_xlim(0.45, 0.68); ax.set_xlabel("Best external validation C-index", fontsize=10)
ax.set_title("A. Ranked biomarker signals\n(replication-consistency first)", fontsize=11, fontweight="bold")
for y, (_, row) in zip(ypos, r.iterrows()):
    if pd.isna(row["cidx_plot"]):
        ax.text(0.505, y, "descriptive\n(no per-patient C)", va="center", ha="left", fontsize=7.3, color=PHY["black"])
    else:
        ax.text(row["cidx_plot"]+0.004, y, f"{row['cidx_plot']:.3f}", va="center", fontsize=8.3)
ax.text(0.5, -1.15, "dashed = 0.50 (random). All signals discovery_only.",
        transform=ax.get_yaxis_transform() if False else ax.transData, fontsize=7)
# legend
from matplotlib.patches import Patch
leg = [Patch(facecolor=PHY["orange"], edgecolor="k", label="Replicates 1 of 2 cohorts"),
       Patch(facecolor=PHY["green"], edgecolor="k", label="Near-null (all cohorts)"),
       Patch(facecolor=PHY["cream"], edgecolor="k", label="Descriptive only")]
ax.legend(handles=leg, fontsize=7.2, loc="lower right", frameon=True)

# ---------- Panel B: dx vs relapse breakpoints (all 5 subjects) ----------
ax = axes[1]
a = arch.sort_values("breakpoint_delta", ascending=True).reset_index(drop=True)
y = np.arange(len(a))
ax.hlines(y, a["dx_breakpoints"], a["relapse_breakpoints"],
          color=[sig_color[s] for s in a["architecture_signature"]], lw=3, zorder=1)
ax.scatter(a["dx_breakpoints"], y, color=PHY["cream"], edgecolor=PHY["black"], s=70, zorder=3, label="diagnosis")
ax.scatter(a["relapse_breakpoints"], y, color=PHY["black"], s=70, zorder=3, label="relapse")
for yi, (_, row) in zip(y, a.iterrows()):
    d = int(row["breakpoint_delta"])
    star = "  *S260" if row["subject"] == 260 else ""
    ax.text(max(row["dx_breakpoints"], row["relapse_breakpoints"])+8, yi,
            f"Δ{'+' if d>=0 else ''}{d}{star}", va="center", fontsize=8.5,
            fontweight="bold" if row["subject"]==260 else "normal")
ax.set_yticks(y); ax.set_yticklabels([f"Subj {int(s)}" for s in a["subject"]], fontsize=9)
ax.set_xlabel("Breakpoint count (segmented CN, BP_THR=0.10)", fontsize=10)
ax.set_title("B. Breakpoint burden: diagnosis → relapse\n(5 subjects, real QDNAseq hg19 30kb)", fontsize=11, fontweight="bold")
ax.set_xlim(-10, 430)
ax.legend(fontsize=8, loc="lower right", frameon=True)

# ---------- Panel C: architecture signature (dispersion vs focality) ----------
ax = axes[2]
a2 = arch.copy()
sizes = 40 + a2["net_gained_breakpoints"].clip(lower=0) * 1.4
sc_colors = [sig_color[s] for s in a2["architecture_signature"]]
ax.scatter(a2["n_chromosomes_gained"], a2["top_focus_fraction"], s=sizes,
           c=sc_colors, edgecolor=PHY["black"], alpha=0.9, zorder=3)
for _, row in a2.iterrows():
    ax.annotate(f"S{int(row['subject'])}",
                (row["n_chromosomes_gained"], row["top_focus_fraction"]),
                textcoords="offset points", xytext=(7,4),
                fontsize=9, fontweight="bold" if row["subject"]==260 else "normal")
# focal-catastrophe corner guide
ax.axhline(0.4, color=PHY["pink"], ls="--", lw=1.1, alpha=0.8)
ax.axvline(8, color=PHY["blue"], ls="--", lw=1.1, alpha=0.6)
ax.text(1.2, 0.42, "focal-catastrophe zone\n(top-frac ≥ 0.4)", fontsize=7.3, color=PHY["pink"], va="bottom")
ax.text(15.5, 0.02, "dispersed CIN zone\n(≥8 chroms, low focality)", fontsize=7.3, color=PHY["blue"], ha="right")
ax.set_xlabel("Chromosomes gaining breakpoints", fontsize=10)
ax.set_ylabel("Top-chromosome focus fraction", fontsize=10)
ax.set_title("C. Rearrangement geometry\n(dispersed vs focal; size ∝ net gained)", fontsize=11, fontweight="bold")
ax.set_xlim(0, 24); ax.set_ylim(0, 0.65)
leg2 = [Patch(facecolor=PHY["blue"], edgecolor="k", label="dispersed_stochastic_CIN"),
        Patch(facecolor=PHY["orange"], edgecolor="k", label="intermediate_rearrangement"),
        Patch(facecolor=PHY["green"], edgecolor="k", label="modest_gain")]
ax.legend(handles=leg2, fontsize=7.2, loc="upper right", frameon=True)

fig.suptitle("Biomarker subsystem — real-data validation (BriTROC-1 sWGS, n=5 dx/relapse pairs) + external cohort ranking",
             fontsize=12.5, fontweight="bold", y=0.985)

png = f"{R}/fig_ranked_signals_and_breakpoints.png"
svg = f"{R}/fig_ranked_signals_and_breakpoints.svg"
fig.savefig(png, dpi=160, bbox_inches="tight")
fig.savefig(svg, bbox_inches="tight")
plt.close(fig)
print("Wrote:", png)
print("Wrote:", svg)
import os
print("PNG bytes:", os.path.getsize(png))


Wrote: /mnt/results/biomarker_subsystem/real_data_validation/fig_ranked_signals_and_breakpoints.png
Wrote: /mnt/results/biomarker_subsystem/real_data_validation/fig_ranked_signals_and_breakpoints.svg
PNG bytes: 212102


Fix label overlaps in breakpoint figure

In [33]:
import matplotlib
matplotlib.rcParams['font.family'] = ['Liberation Sans', 'Arimo', 'DejaVu Sans']
matplotlib.rcParams['svg.fonttype'] = 'none'
import matplotlib.pyplot as plt
import pandas as pd, numpy as np, os
from matplotlib.patches import Patch

W = "/mnt/shared-workspace/shared/data_acq/w3"
R = "/mnt/results/biomarker_subsystem/real_data_validation"
arch = pd.read_csv(f"{W}/subj260_analysis/breakpoint_architecture_with_subj260.csv")
ranked = pd.read_csv(f"{W}/report_assets/ranked_signal_table.csv")

PHY = dict(black="#000000", cream="#ECE9E2", ivory="#FAF9F3", yellow="#E9ED4C",
           orange="#FF9400", green="#75A025", pink="#FD9BED", blue="#0279EE")
sig_color = {"dispersed_stochastic_CIN": PHY["blue"], "intermediate_rearrangement": PHY["orange"],
             "modest_gain": PHY["green"], "chromothripsis_like_focal_catastrophe": PHY["pink"]}

fig, axes = plt.subplots(1, 3, figsize=(17.5, 5.4))
fig.subplots_adjust(wspace=0.46, left=0.075, right=0.985, top=0.86, bottom=0.16)

# Panel A (wider label window + more left margin)
ax = axes[0]
r = ranked.copy()
r["label"] = r["rank"].astype(str) + ". " + r["signal"].str.replace(r"\s*\(.*\)","",regex=True).str[:32]
r["cidx_plot"] = pd.to_numeric(r["best_ext_cindex"], errors="coerce")
ypos = np.arange(len(r))[::-1]
colors = [PHY["cream"] if pd.isna(v) else (PHY["orange"] if rep=="REPLICATES_1of2" else PHY["green"])
          for v, rep in zip(r["cidx_plot"], r["replication"])]
ax.barh(ypos, r["cidx_plot"].fillna(0.5), color=colors, edgecolor=PHY["black"], height=0.6)
ax.axvline(0.5, color=PHY["black"], ls=":", lw=1.2, alpha=0.7)
ax.set_yticks(ypos); ax.set_yticklabels(r["label"], fontsize=9)
ax.set_xlim(0.45, 0.68); ax.set_xlabel("Best external validation C-index", fontsize=10)
ax.set_title("A. Ranked biomarker signals\n(replication-consistency first)", fontsize=11, fontweight="bold")
for y, (_, row) in zip(ypos, r.iterrows()):
    if pd.isna(row["cidx_plot"]):
        ax.text(0.505, y, "descriptive\n(no per-patient C)", va="center", ha="left", fontsize=7.3)
    else:
        ax.text(row["cidx_plot"]+0.004, y, f"{row['cidx_plot']:.3f}", va="center", fontsize=8.3)
ax.legend(handles=[Patch(facecolor=PHY["orange"], edgecolor="k", label="Replicates 1 of 2 cohorts"),
                   Patch(facecolor=PHY["green"], edgecolor="k", label="Near-null (all cohorts)"),
                   Patch(facecolor=PHY["cream"], edgecolor="k", label="Descriptive only")],
          fontsize=7.2, loc="lower right", frameon=True)

# Panel B (unchanged)
ax = axes[1]
a = arch.sort_values("breakpoint_delta", ascending=True).reset_index(drop=True)
y = np.arange(len(a))
ax.hlines(y, a["dx_breakpoints"], a["relapse_breakpoints"],
          color=[sig_color[s] for s in a["architecture_signature"]], lw=3, zorder=1)
ax.scatter(a["dx_breakpoints"], y, color=PHY["cream"], edgecolor=PHY["black"], s=70, zorder=3, label="diagnosis")
ax.scatter(a["relapse_breakpoints"], y, color=PHY["black"], s=70, zorder=3, label="relapse")
for yi, (_, row) in zip(y, a.iterrows()):
    d = int(row["breakpoint_delta"]); star = "  *S260" if row["subject"]==260 else ""
    ax.text(max(row["dx_breakpoints"], row["relapse_breakpoints"])+8, yi,
            f"Δ{'+' if d>=0 else ''}{d}{star}", va="center", fontsize=8.5,
            fontweight="bold" if row["subject"]==260 else "normal")
ax.set_yticks(y); ax.set_yticklabels([f"Subj {int(s)}" for s in a["subject"]], fontsize=9)
ax.set_xlabel("Breakpoint count (segmented CN, BP_THR=0.10)", fontsize=10)
ax.set_title("B. Breakpoint burden: diagnosis → relapse\n(5 subjects, real QDNAseq hg19 30kb)", fontsize=11, fontweight="bold")
ax.set_xlim(-10, 430); ax.legend(fontsize=8, loc="lower right", frameon=True)

# Panel C (nudge overlapping labels apart via per-subject offsets)
ax = axes[2]
a2 = arch.copy()
sizes = 40 + a2["net_gained_breakpoints"].clip(lower=0) * 1.4
ax.scatter(a2["n_chromosomes_gained"], a2["top_focus_fraction"], s=sizes,
           c=[sig_color[s] for s in a2["architecture_signature"]], edgecolor=PHY["black"], alpha=0.9, zorder=3)
# custom label offsets to avoid S6/S191 collision (both high chrom-count, low focus)
offsets = {6:(7,-13), 191:(7,7), 260:(7,6), 119:(8,4), 109:(-22,4)}
for _, row in a2.iterrows():
    s = int(row["subject"]); ox, oy = offsets.get(s, (7,4))
    ax.annotate(f"S{s}", (row["n_chromosomes_gained"], row["top_focus_fraction"]),
                textcoords="offset points", xytext=(ox,oy),
                fontsize=9, fontweight="bold" if s==260 else "normal")
ax.axhline(0.4, color=PHY["pink"], ls="--", lw=1.1, alpha=0.8)
ax.axvline(8, color=PHY["blue"], ls="--", lw=1.1, alpha=0.6)
ax.text(1.2, 0.42, "focal-catastrophe zone\n(top-frac ≥ 0.4)", fontsize=7.3, color=PHY["pink"], va="bottom")
ax.text(15.5, 0.02, "dispersed CIN zone\n(≥8 chroms, low focality)", fontsize=7.3, color=PHY["blue"], ha="right")
ax.set_xlabel("Chromosomes gaining breakpoints", fontsize=10)
ax.set_ylabel("Top-chromosome focus fraction", fontsize=10)
ax.set_title("C. Rearrangement geometry\n(dispersed vs focal; size ∝ net gained)", fontsize=11, fontweight="bold")
ax.set_xlim(0, 24); ax.set_ylim(0, 0.65)
ax.legend(handles=[Patch(facecolor=PHY["blue"], edgecolor="k", label="dispersed_stochastic_CIN"),
                   Patch(facecolor=PHY["orange"], edgecolor="k", label="intermediate_rearrangement"),
                   Patch(facecolor=PHY["green"], edgecolor="k", label="modest_gain")],
          fontsize=7.2, loc="upper right", frameon=True)

fig.suptitle("Biomarker subsystem — real-data validation (BriTROC-1 sWGS, n=5 dx/relapse pairs) + external cohort ranking",
             fontsize=12.5, fontweight="bold", y=0.985)
png = f"{R}/fig_ranked_signals_and_breakpoints.png"; svg = f"{R}/fig_ranked_signals_and_breakpoints.svg"
fig.savefig(png, dpi=160, bbox_inches="tight"); fig.savefig(svg, bbox_inches="tight"); plt.close(fig)
print("Rewrote figure. PNG bytes:", os.path.getsize(png))


Rewrote figure. PNG bytes: 198742


Add leader lines to Panel C to resolve label overlap

In [35]:
import matplotlib
matplotlib.rcParams['font.family'] = ['Liberation Sans', 'Arimo', 'DejaVu Sans']
matplotlib.rcParams['svg.fonttype'] = 'none'
import matplotlib.pyplot as plt
import pandas as pd, numpy as np, os
from matplotlib.patches import Patch

W = "/mnt/shared-workspace/shared/data_acq/w3"
R = "/mnt/results/biomarker_subsystem/real_data_validation"
arch = pd.read_csv(f"{W}/subj260_analysis/breakpoint_architecture_with_subj260.csv")
ranked = pd.read_csv(f"{W}/report_assets/ranked_signal_table.csv")
PHY = dict(black="#000000", cream="#ECE9E2", ivory="#FAF9F3", yellow="#E9ED4C",
           orange="#FF9400", green="#75A025", pink="#FD9BED", blue="#0279EE")
sig_color = {"dispersed_stochastic_CIN": PHY["blue"], "intermediate_rearrangement": PHY["orange"],
             "modest_gain": PHY["green"], "chromothripsis_like_focal_catastrophe": PHY["pink"]}

fig, axes = plt.subplots(1, 3, figsize=(17.5, 5.4))
fig.subplots_adjust(wspace=0.46, left=0.075, right=0.985, top=0.86, bottom=0.16)

# Panel A
ax = axes[0]
r = ranked.copy()
r["label"] = r["rank"].astype(str) + ". " + r["signal"].str.replace(r"\s*\(.*\)","",regex=True).str[:32]
r["cidx_plot"] = pd.to_numeric(r["best_ext_cindex"], errors="coerce")
ypos = np.arange(len(r))[::-1]
colors = [PHY["cream"] if pd.isna(v) else (PHY["orange"] if rep=="REPLICATES_1of2" else PHY["green"])
          for v, rep in zip(r["cidx_plot"], r["replication"])]
ax.barh(ypos, r["cidx_plot"].fillna(0.5), color=colors, edgecolor=PHY["black"], height=0.6)
ax.axvline(0.5, color=PHY["black"], ls=":", lw=1.2, alpha=0.7)
ax.set_yticks(ypos); ax.set_yticklabels(r["label"], fontsize=9)
ax.set_xlim(0.45, 0.68); ax.set_xlabel("Best external validation C-index", fontsize=10)
ax.set_title("A. Ranked biomarker signals\n(replication-consistency first)", fontsize=11, fontweight="bold")
for y, (_, row) in zip(ypos, r.iterrows()):
    if pd.isna(row["cidx_plot"]):
        ax.text(0.505, y, "descriptive\n(no per-patient C)", va="center", ha="left", fontsize=7.3)
    else:
        ax.text(row["cidx_plot"]+0.004, y, f"{row['cidx_plot']:.3f}", va="center", fontsize=8.3)
ax.legend(handles=[Patch(facecolor=PHY["orange"], edgecolor="k", label="Replicates 1 of 2 cohorts"),
                   Patch(facecolor=PHY["green"], edgecolor="k", label="Near-null (all cohorts)"),
                   Patch(facecolor=PHY["cream"], edgecolor="k", label="Descriptive only")],
          fontsize=7.2, loc="lower right", frameon=True)

# Panel B (offset S260 delta label upward to avoid crowding relapse marker)
ax = axes[1]
a = arch.sort_values("breakpoint_delta", ascending=True).reset_index(drop=True)
y = np.arange(len(a))
ax.hlines(y, a["dx_breakpoints"], a["relapse_breakpoints"],
          color=[sig_color[s] for s in a["architecture_signature"]], lw=3, zorder=1)
ax.scatter(a["dx_breakpoints"], y, color=PHY["cream"], edgecolor=PHY["black"], s=70, zorder=3, label="diagnosis")
ax.scatter(a["relapse_breakpoints"], y, color=PHY["black"], s=70, zorder=3, label="relapse")
for yi, (_, row) in zip(y, a.iterrows()):
    d = int(row["breakpoint_delta"])
    if row["subject"]==260:
        ax.text(row["relapse_breakpoints"]+14, yi+0.16, f"Δ+{d}  *S260", va="center",
                fontsize=8.5, fontweight="bold")
    else:
        ax.text(max(row["dx_breakpoints"], row["relapse_breakpoints"])+8, yi,
                f"Δ{'+' if d>=0 else ''}{d}", va="center", fontsize=8.5)
ax.set_yticks(y); ax.set_yticklabels([f"Subj {int(s)}" for s in a["subject"]], fontsize=9)
ax.set_xlabel("Breakpoint count (segmented CN, BP_THR=0.10)", fontsize=10)
ax.set_title("B. Breakpoint burden: diagnosis → relapse\n(5 subjects, real QDNAseq hg19 30kb)", fontsize=11, fontweight="bold")
ax.set_xlim(-10, 440); ax.legend(fontsize=8, loc="lower right", frameon=True)

# Panel C — leader-line annotations for the overlapping S6/S191 cluster; lower alpha
ax = axes[2]
a2 = arch.copy()
sizes = 40 + a2["net_gained_breakpoints"].clip(lower=0) * 1.4
ax.scatter(a2["n_chromosomes_gained"], a2["top_focus_fraction"], s=sizes,
           c=[sig_color[s] for s in a2["architecture_signature"]], edgecolor=PHY["black"],
           alpha=0.75, zorder=3, linewidths=1.1)
# text targets (data coords) pulled clear, with arrows back to points
label_targets = {
    191: (16.5, 0.30),   # pull up-left, away from cluster
    6:   (20.5, 0.235),  # pull up
    260: (16.8, 0.155),  # its own spot
    119: (11.6, 0.315),
    109: (5.3, 0.60),
}
for _, row in a2.iterrows():
    s = int(row["subject"]); tx, ty = label_targets[s]
    ax.annotate(f"S{s}", xy=(row["n_chromosomes_gained"], row["top_focus_fraction"]),
                xytext=(tx, ty), fontsize=9, fontweight="bold" if s==260 else "normal",
                ha="center",
                arrowprops=dict(arrowstyle="-", color=PHY["black"], lw=0.7, alpha=0.7,
                                shrinkA=1, shrinkB=4))
ax.axhline(0.4, color=PHY["pink"], ls="--", lw=1.1, alpha=0.8)
ax.axvline(8, color=PHY["blue"], ls="--", lw=1.1, alpha=0.6)
ax.text(1.2, 0.42, "focal-catastrophe zone\n(top-frac ≥ 0.4)", fontsize=7.3, color=PHY["pink"], va="bottom")
ax.text(15.5, 0.015, "dispersed CIN zone\n(≥8 chroms, low focality)", fontsize=7.3, color=PHY["blue"], ha="right")
ax.set_xlabel("Chromosomes gaining breakpoints", fontsize=10)
ax.set_ylabel("Top-chromosome focus fraction", fontsize=10)
ax.set_title("C. Rearrangement geometry\n(dispersed vs focal; size ∝ net gained)", fontsize=11, fontweight="bold")
ax.set_xlim(0, 24); ax.set_ylim(0, 0.65)
ax.legend(handles=[Patch(facecolor=PHY["blue"], edgecolor="k", label="dispersed_stochastic_CIN"),
                   Patch(facecolor=PHY["orange"], edgecolor="k", label="intermediate_rearrangement"),
                   Patch(facecolor=PHY["green"], edgecolor="k", label="modest_gain")],
          fontsize=7.2, loc="upper right", frameon=True)

fig.suptitle("Biomarker subsystem — real-data validation (BriTROC-1 sWGS, n=5 dx/relapse pairs) + external cohort ranking",
             fontsize=12.5, fontweight="bold", y=0.985)
png = f"{R}/fig_ranked_signals_and_breakpoints.png"; svg = f"{R}/fig_ranked_signals_and_breakpoints.svg"
fig.savefig(png, dpi=160, bbox_inches="tight"); fig.savefig(svg, bbox_inches="tight"); plt.close(fig)
print("Rewrote figure with leader lines. PNG bytes:", os.path.getsize(png))


Rewrote figure with leader lines. PNG bytes: 204249


Update reconciliation ledger items 1 and 5 for n=5

In [37]:
import pandas as pd

R = "/mnt/results/biomarker_subsystem/real_data_validation/switch_reconciliation_ledger.csv"
df = pd.read_csv(R)
print("cols:", list(df.columns)); print("rows:", len(df))

# Item 1: strengthen evidence with Subject 260 corroboration (n=5), decline UNCHANGED
m1 = df["item"].str.startswith("1. DATA-TO-LOGIT")
df.loc[m1, "evidence"] = ("n=1 cannot identify a multi-coefficient logit (overfitting, not calibration). "
    "Pilot produced COPY-NUMBER, not CA-125/KELIM kinetics/outcomes -- not this logit's input. Prior turn "
    "set RES-OV-P0-005 -> DISCOVERY_ONLY_NOT_CALIBRATED. This-session bp_arch confirms Subj191 delta=+253; "
    "Subject 260 pulled+processed this session confirms delta=+60 (net +66), a 5th real data point that "
    "CORROBORATES the structural-divergence FEATURE but still does not calibrate an outcome logit.")
df.loc[m1, "honest_alternative"] = ("Subj191 +253 (and Subj260 +60) documented as Serial-SAE FEATURE, wired as "
    "bounded-monotone heuristic (Signal 8), conf<=0.40, PILOT_DEMONSTRATION_N5_NOT_CLINICAL. Real calibration "
    "needs a serial CA-125 outcome cohort (ARIEL2/3, GOG-0218), access-gated.")

# Item 5: n=4 -> n=5; Subject 260 gene CN now also computed (adds PTEN neutral->loss)
m5 = df["item"].str.startswith("5. GENE UNIVERSE")
df.loc[m5, "evidence"] = ("Targeted relative-CN readout run THIS SESSION on all 5 genes + CCNE1 from real "
    "pilot segmented CN (4 pilot subjects), and independently on Subject 260 (dx IM_426/relapse JBLAB-19323). "
    "CN metric now EXISTS for every gene across 5 subjects. But n=5=pilot; ~0.1x sWGS/30kb detects LARGE CNA "
    "only, cannot call the SNV/indel LoF driving KEAP1/SLC31A1/PALB2; most calls neutral. Subject 260 does show "
    "an ACQUIRED PTEN loss at relapse (0.967->0.692) -- hypothesis-grade, not a validated call.")
df.loc[m5, "honest_alternative"] = ("Per-gene verdict: PTEN/PALB2/CUL3 = CANDIDATE (focal loss, needs cohort); "
    "KEAP1/SLC31A1 stay LITERATURE_ONLY (SNV mechanism, not sWGS-callable). 0/5 reach VALIDATED from n=5. Raw "
    "sequence DID give a CN metric; did NOT give the mutation metric these genes need.")

# Exec-mandate row: v3 10/0 note stays; no change needed (still cannot verify blanket flip)
df.to_csv(R, index=False)
print("\nUpdated items 1 and 5 for n=5 (Subject 260 processed). Decline verdicts unchanged.")
print("\n=== item 1 status:", df.loc[m1,"actual_status"].iloc[0])
print("=== item 5 status:", df.loc[m5,"actual_status"].iloc[0])

# also refresh the copy in shared workspace so both locations agree
import shutil
shutil.copy(R, "/mnt/shared-workspace/shared/data_acq/w3/ledger/switch_reconciliation_ledger.csv")
print("\nSynced to shared workspace ledger dir.")


cols: ['item', 'actual_status', 'evidence', 'honest_alternative']
rows: 8

Updated items 1 and 5 for n=5 (Subject 260 processed). Decline verdicts unchanged.

=== item 1 status: DECLINED_FABRICATION
=== item 5 status: DONE_THIS_SESSION + HONEST_PARTIAL

Synced to shared workspace ledger dir.


Load exact verified numbers from receipt files

In [39]:
import pandas as pd, json

R = "/mnt/results/biomarker_subsystem/real_data_validation"

# --- Merged 5-subject architecture table (the evidentiary backbone) ---
arch = pd.read_csv(f"{R}/breakpoint_architecture_with_subj260.csv")
print("=== 5-subject breakpoint architecture (verbatim from disk) ===")
print(arch.to_string(index=False))

# --- Subject 260 meta (lead subject provenance + QC + per-chrom) ---
meta = json.load(open(f"{R}/subj260_analysis_meta.json"))
print("\n=== Subject 260 architecture block ===")
for k,v in meta["architecture"].items(): print(f"  {k}: {v}")
print("\n=== Subject 260 QC ===")
for q in meta["qc"]: print(" ", q)
print("\n=== Subject 260 per-chromosome delta (nonzero) ===")
print(" ", meta["delta_by_chrom"])
print("\n  user_asserted_delta:", meta["user_asserted_delta"])
print("  computed_breakpoint_delta:", meta["computed_breakpoint_delta"])
print("  honest_ceiling:", meta["honest_ceiling"])

# --- Subject 260 gene CN dx vs relapse ---
g = pd.read_csv(f"{R}/subj260_gene_cn_dx_vs_relapse.csv")
print("\n=== Subject 260 gene CN dx->relapse ===")
print(g.to_string(index=False))


=== 5-subject breakpoint architecture (verbatim from disk) ===
 subject dx_sample relapse_sample  dx_breakpoints  relapse_breakpoints  breakpoint_delta  net_gained_breakpoints  n_chromosomes_gained  top_focus_chrom  top_focus_fraction  top_chrom_cn_levels     architecture_signature
     191    IM_280     JBLAB-4944             127                  380               253                     253                    21                1               0.115                    5   dispersed_stochastic_CIN
       6    IM_274     JBLAB-4110              17                  118               101                     101                    20                8               0.119                    9   dispersed_stochastic_CIN
     260    IM_426    JBLAB-19323              75                  135                60                      66                    16                1               0.212                    5   dispersed_stochastic_CIN
     119    IM_249     JBLAB-4232             148        

Loading W2 Riester replication for forest plot

In [41]:
import json, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = ['Liberation Sans', 'Arimo', 'DejaVu Sans']
matplotlib.rcParams['svg.fonttype'] = 'none'

REAUDIT = "/mnt/shared-workspace/shared/biomarker_reaudit"
FIGDIR = "/mnt/results/biomarker_subsystem/figures"

# Phylo palette
PHY = dict(black="#000000", cream="#ECE9E2", offwhite="#FAF9F3", yellow="#E9ED4C",
           orange="#FF9400", green="#75A025", pink="#FD9BED", blue="#0279EE")

# ---- Load W2 Riester replication ----
w2 = json.load(open(f"{REAUDIT}/w2_prognostic_replication_reaudit.json"))
rc = w2["replication_cohorts"]
rows = []
for cid, d in rc.items():
    if "c_index" in d and "skipped" not in d and "error" not in d:
        rows.append(dict(cohort=cid, n=d["n"], ev=d["n_events"], c=d["c_index"],
                         lo=d["c_index_lower"], hi=d["c_index_upper"], p=d["c_index_p"],
                         hr=d["hr_per_1sd"], hrlo=d["hr_lower"], hrhi=d["hr_upper"], hrp=d["hr_p"]))
# order by C-index descending
rows.sort(key=lambda r: r["c"], reverse=True)
print("W2 Riester adequate-power cohorts plotted:", len(rows))
def strict(r):
    return (r["lo"] > 0.5) and ((r["hrlo"] > 1.0) or (r["hrhi"] < 1.0))
for r in rows:
    tag = "STRICT" if strict(r) else ""
    print(f"  {r['cohort']:<14} n={r['n']:>3} ev={r['ev']:>3} C={r['c']:.3f} [{r['lo']:.3f}-{r['hi']:.3f}] p={r['p']:.3g} {tag}")


W2 Riester adequate-power cohorts plotted: 15
  GSE9891        n=268 ev=113 C=0.625 [0.569-0.681] p=1.2e-05 STRICT
  GSE13876       n=157 ev=113 C=0.557 [0.499-0.614] p=0.0536 
  GSE26712       n=185 ev=129 C=0.554 [0.502-0.607] p=0.0424 
  GSE14764       n= 79 ev= 21 C=0.552 [0.421-0.683] p=0.439 
  GSE18520       n= 53 ev= 41 C=0.551 [0.466-0.636] p=0.239 
  PMID17290060   n=117 ev= 67 C=0.551 [0.475-0.626] p=0.189 
  GSE51088       n=152 ev=112 C=0.533 [0.481-0.585] p=0.219 
  GSE30161       n= 58 ev= 36 C=0.531 [0.429-0.633] p=0.55 
  E.MTAB.386     n=129 ev= 73 C=0.527 [0.446-0.608] p=0.517 
  GSE17260       n=110 ev= 46 C=0.514 [0.417-0.610] p=0.778 
  GSE32062       n=257 ev=120 C=0.509 [0.451-0.567] p=0.767 
  GSE32063       n= 40 ev= 22 C=0.508 [0.352-0.663] p=0.924 
  PMID19318476   n= 42 ev= 22 C=0.494 [0.370-0.618] p=0.926 
  GSE8842        n= 82 ev= 15 C=0.475 [0.340-0.610] p=0.716 
  GSE49997       n=194 ev= 57 C=0.465 [0.378-0.551] p=0.426 


Building Riester 15-cohort C-index forest plot

In [43]:
import matplotlib.patches as mpatches
# ---- Figure: Riester 15-cohort C-index forest ----
fig, ax = plt.subplots(figsize=(8.2, 6.4))
ys = np.arange(len(rows))[::-1]  # top = best
for y, r in zip(ys, rows):
    is_strict = strict(r)
    col = PHY["orange"] if is_strict else PHY["blue"]
    lw = 2.6 if is_strict else 1.6
    ax.plot([r["lo"], r["hi"]], [y, y], color=col, lw=lw, solid_capstyle="round", zorder=2)
    ax.scatter([r["c"]], [y], s=(70 if is_strict else 42), color=col,
               edgecolor=PHY["black"], linewidth=0.8, zorder=3)

ax.axvline(0.5, color=PHY["black"], lw=1.2, ls="--", zorder=1)
ax.set_yticks(ys)
ax.set_yticklabels([f"{r['cohort']}  (n={r['n']}, e={r['ev']})" for r in rows], fontsize=9)
ax.set_xlabel("Concordance index (C-index), 95% CI", fontsize=11)
ax.set_xlim(0.30, 0.72)
ax.set_title("Riester 3-gene prognostic score — external replication\nFrozen coefficients applied to 15 adequate-power MetaGxOvarian cohorts",
             fontsize=11.5, loc="left")

# annotate GSE9891 strict signal
top = rows[0]
ax.annotate(f"C={top['c']:.3f}\np={top['p']:.1e}", xy=(top["hi"], ys[0]),
            xytext=(top["hi"]+0.012, ys[0]), va="center", fontsize=8.5,
            color=PHY["orange"], fontweight="bold")

strict_patch = mpatches.Patch(color=PHY["orange"], label="Strict signal (C-index CI>0.5 & HR CI excludes 1)")
weak_patch = mpatches.Patch(color=PHY["blue"], label="Does not clear strict bar")
ax.legend(handles=[strict_patch, weak_patch], loc="lower right", fontsize=8.2, frameon=True)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

png = f"{FIGDIR}/fig_v4_riester_replication_forest.png"
svg = f"{FIGDIR}/fig_v4_riester_replication_forest.svg"
fig.savefig(png, dpi=200, bbox_inches="tight")
fig.savefig(svg, bbox_inches="tight")
plt.close(fig)
print("saved:", png.split("/")[-1], "and", svg.split("/")[-1])


saved: fig_v4_riester_replication_forest.png and fig_v4_riester_replication_forest.svg


Building DDR + PI3K/mTOR 17-cohort two-panel forest

In [45]:
# ---- Load W4 scan ----
w4 = json.load(open(f"{REAUDIT}/w4_baseline_scan_reaudit.json"))

def panel_rows(panel_key):
    pc = w4["panels"][panel_key]["per_cohort"]
    out = []
    for cid, d in pc.items():
        if "c_index" in d and "skipped" not in d:
            out.append(dict(cohort=cid, n=d["n"], ev=d["n_events"], c=d["c_index"],
                            lo=d["c_index_lower"], hi=d["c_index_upper"], p=d["c_index_p"],
                            hrlo=d["hr_lower"], hrhi=d["hr_upper"]))
    out.sort(key=lambda r: r["c"], reverse=True)
    return out

ddr = panel_rows("ddr")
pi3k = panel_rows("pi3k_mtor")
print("DDR cohorts:", len(ddr), "| PI3K cohorts:", len(pi3k))
print("DDR strict:", w4["panels"]["ddr"]["cohorts_strict_signal"], "verdict:", w4["panels"]["ddr"]["verdict"])
print("PI3K strict:", w4["panels"]["pi3k_mtor"]["cohorts_strict_signal"], "verdict:", w4["panels"]["pi3k_mtor"]["verdict"])

def strict4(r):
    return (r["lo"] > 0.5) and ((r["hrlo"] > 1.0) or (r["hrhi"] < 1.0))
def inverse(r):
    return (r["hi"] < 0.5) and (r["hrhi"] < 1.0)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 6.6), sharex=True)
for ax, data, title, verdict in [
    (axes[0], ddr, "DDR panel", "CONSIDER (1/17 strict)"),
    (axes[1], pi3k, "PI3K / mTOR panel", "discovery_only (0/17 strict, confirmed null)")]:
    ys = np.arange(len(data))[::-1]
    for y, r in zip(ys, data):
        if strict4(r):
            col, lw, s = PHY["orange"], 2.6, 70
        elif inverse(r):
            col, lw, s = PHY["pink"], 2.4, 60
        else:
            col, lw, s = PHY["blue"], 1.6, 42
        ax.plot([r["lo"], r["hi"]], [y, y], color=col, lw=lw, solid_capstyle="round", zorder=2)
        ax.scatter([r["c"]], [y], s=s, color=col, edgecolor=PHY["black"], linewidth=0.8, zorder=3)
    ax.axvline(0.5, color=PHY["black"], lw=1.2, ls="--", zorder=1)
    ax.set_yticks(ys)
    ax.set_yticklabels([f"{r['cohort']} (n={r['n']}, e={r['ev']})" for r in data], fontsize=8.3)
    ax.set_xlabel("C-index, 95% CI", fontsize=11)
    ax.set_title(f"{title}\n{verdict}", fontsize=11, loc="left")
    ax.set_xlim(0.28, 0.80)
    ax.spines[["top", "right"]].set_visible(False)

strict_patch = mpatches.Patch(color=PHY["orange"], label="Strict signal (CI>0.5 & HR excludes 1)")
inv_patch = mpatches.Patch(color=PHY["pink"], label="Significant inverse (CI<0.5)")
weak_patch = mpatches.Patch(color=PHY["blue"], label="No strict signal")
fig.legend(handles=[strict_patch, inv_patch, weak_patch], loc="lower center",
           ncol=3, fontsize=8.6, frameon=True, bbox_to_anchor=(0.5, -0.03))
fig.suptitle("Baseline discovery scan — unsupervised mean-z panel scores across 17 adequate-power cohorts",
             fontsize=12, x=0.02, ha="left")
plt.tight_layout(rect=[0, 0.02, 1, 0.97])

png = f"{FIGDIR}/fig_v4_ddr_pi3k_panel_forest.png"
svg = f"{FIGDIR}/fig_v4_ddr_pi3k_panel_forest.svg"
fig.savefig(png, dpi=200, bbox_inches="tight")
fig.savefig(svg, bbox_inches="tight")
plt.close(fig)
print("saved:", png.split("/")[-1], "and", svg.split("/")[-1])


DDR cohorts: 17 | PI3K cohorts: 17
DDR strict: GSE17260 verdict: CONSIDER
PI3K strict: {} verdict: discovery_only
saved: fig_v4_ddr_pi3k_panel_forest.png and fig_v4_ddr_pi3k_panel_forest.svg


Verifying checker concerns against receipt data

In [47]:
# Verify the checker's concerns against the data
print("=== Row counts ===")
print("DDR rows:", len(ddr), "| PI3K rows:", len(pi3k))
print("DDR cohorts:", [r["cohort"] for r in ddr])
print("PI3K cohorts:", [r["cohort"] for r in pi3k])

print("\n=== PMID17290060 in each panel (strict inverse test: hi<0.5 AND hrhi<1) ===")
for name, data in [("DDR", ddr), ("PI3K", pi3k)]:
    r = [x for x in data if x["cohort"]=="PMID17290060"][0]
    print(f"{name}: C={r['c']:.3f} CI[{r['lo']:.3f}-{r['hi']:.3f}] HR_CI[{r['hrlo']:.3f}-{r['hrhi']:.3f}] "
          f"-> inverse={inverse(r)} (hi<0.5={r['hi']<0.5}, hrhi<1={r['hrhi']<1.0})")

print("\n=== GSE8842 (potential clip) ===")
for name, data in [("DDR", ddr), ("PI3K", pi3k)]:
    m = [x for x in data if x["cohort"]=="GSE8842"]
    if m:
        r=m[0]; print(f"{name}: C={r['c']:.3f} CI[{r['lo']:.3f}-{r['hi']:.3f}]  (xlim starts 0.28)")
print("\nMin lower-CI across both panels:", min([r['lo'] for r in ddr+pi3k]))


=== Row counts ===
DDR rows: 17 | PI3K rows: 17
DDR cohorts: ['GSE17260', 'GSE32062', 'GSE13876', 'TCGAOVARIAN', 'GSE51088', 'GSE18520', 'GSE9891', 'GSE14764', 'PMID19318476', 'GSE26712', 'E.MTAB.386', 'GSE30009', 'GSE49997', 'GSE8842', 'GSE32063', 'GSE30161', 'PMID17290060']
PI3K cohorts: ['GSE14764', 'GSE9891', 'GSE13876', 'TCGAOVARIAN', 'GSE17260', 'GSE32062', 'GSE49997', 'PMID19318476', 'GSE30161', 'GSE26712', 'GSE51088', 'GSE8842', 'E.MTAB.386', 'GSE30009', 'GSE18520', 'GSE32063', 'PMID17290060']

=== PMID17290060 in each panel (strict inverse test: hi<0.5 AND hrhi<1) ===
DDR: C=0.400 CI[0.330-0.469] HR_CI[0.565-0.945] -> inverse=True (hi<0.5=True, hrhi<1=True)
PI3K: C=0.383 CI[0.311-0.455] HR_CI[0.628-1.039] -> inverse=False (hi<0.5=True, hrhi<1=False)

=== GSE8842 (potential clip) ===
DDR: C=0.438 CI[0.268-0.607]  (xlim starts 0.28)
PI3K: C=0.477 CI[0.341-0.614]  (xlim starts 0.28)

Min lower-CI across both panels: 0.268361


Regenerating DDR/PI3K forest with clip fix and clarifying note

In [49]:
# Regenerate with wider xlim (fix real clip) + clarify PI3K lowest point
fig, axes = plt.subplots(1, 2, figsize=(12.5, 6.6), sharex=True)
for ax, data, title, verdict in [
    (axes[0], ddr, "DDR panel", "CONSIDER (1/17 strict)"),
    (axes[1], pi3k, "PI3K / mTOR panel", "discovery_only (0/17 strict, confirmed null)")]:
    ys = np.arange(len(data))[::-1]
    for y, r in zip(ys, data):
        if strict4(r):
            col, lw, s = PHY["orange"], 2.6, 70
        elif inverse(r):
            col, lw, s = PHY["pink"], 2.4, 60
        else:
            col, lw, s = PHY["blue"], 1.6, 42
        ax.plot([r["lo"], r["hi"]], [y, y], color=col, lw=lw, solid_capstyle="round", zorder=2)
        ax.scatter([r["c"]], [y], s=s, color=col, edgecolor=PHY["black"], linewidth=0.8, zorder=3)
    ax.axvline(0.5, color=PHY["black"], lw=1.2, ls="--", zorder=1)
    ax.set_yticks(ys)
    ax.set_yticklabels([f"{r['cohort']} (n={r['n']}, e={r['ev']})" for r in data], fontsize=8.3)
    ax.set_xlabel("C-index, 95% CI", fontsize=11)
    ax.set_title(f"{title}\n{verdict}", fontsize=11, loc="left")
    ax.set_xlim(0.25, 0.80)   # widened to fully show GSE8842 DDR lower CI (0.268)
    ax.spines[["top", "right"]].set_visible(False)

# Clarify: PI3K PMID17290060 is directionally inverse but HR CI includes 1 (not strict-inverse -> stays blue)
pi3k_worst = pi3k[-1]  # sorted desc, last = lowest C
axr = axes[1]
yrow = 0  # bottom row
axr.annotate("lowest C but HR CI\nincludes 1 (not strict)", xy=(pi3k_worst["lo"], yrow),
             xytext=(0.255, yrow+1.2), fontsize=7.4, color=PHY["black"],
             arrowprops=dict(arrowstyle="->", color=PHY["black"], lw=0.8))

strict_patch = mpatches.Patch(color=PHY["orange"], label="Strict signal (CI>0.5 & HR excludes 1)")
inv_patch = mpatches.Patch(color=PHY["pink"], label="Significant inverse (C-index CI<0.5 & HR excludes 1)")
weak_patch = mpatches.Patch(color=PHY["blue"], label="No strict signal")
fig.legend(handles=[strict_patch, inv_patch, weak_patch], loc="lower center",
           ncol=3, fontsize=8.4, frameon=True, bbox_to_anchor=(0.5, -0.03))
fig.suptitle("Baseline discovery scan — unsupervised mean-z panel scores across 17 adequate-power cohorts",
             fontsize=12, x=0.02, ha="left")
plt.tight_layout(rect=[0, 0.02, 1, 0.97])

png = f"{FIGDIR}/fig_v4_ddr_pi3k_panel_forest.png"
svg = f"{FIGDIR}/fig_v4_ddr_pi3k_panel_forest.svg"
fig.savefig(png, dpi=200, bbox_inches="tight")
fig.savefig(svg, bbox_inches="tight")
plt.close(fig)
print("re-saved with wider xlim + PI3K clarifying annotation")


re-saved with wider xlim + PI3K clarifying annotation


Final cross-check: manuscript headline numbers vs receipts

In [51]:
import json
R = "/mnt/shared-workspace/shared/biomarker_reaudit"
w1 = json.load(open(f"{R}/w1_prognostic_reaudit.json"))
w2 = json.load(open(f"{R}/w2_prognostic_replication_reaudit.json"))
w4 = json.load(open(f"{R}/w4_baseline_scan_reaudit.json"))

checks = []
# W1 OOF
cv = w1["internal_cv"]
checks.append(("W1 OOF C-index=0.539", round(cv["metric_value"],3)==0.539))
checks.append(("W1 OOF CI 0.500-0.579", round(cv["ci95_low"],3)==0.500 and round(cv["ci95_high"],3)==0.579))
checks.append(("W1 OOF p=0.050", round(cv["p_value"],3)==0.050))
# W2 GSE9891
g = w2["replication_cohorts"]["GSE9891"]
checks.append(("W2 GSE9891 C=0.625", round(g["c_index"],3)==0.625))
checks.append(("W2 GSE9891 HR=1.36", round(g["hr_per_1sd"],2)==1.36))
checks.append(("W2 GSE9891 PH p=0.045", round(g["ph_test_p"],3)==0.045))
checks.append(("W2 verdict CONSIDER", w2["verdict"]=="CONSIDER"))
checks.append(("W2 15 indep cohorts", w2["n_independent_cohorts_adequate_power"]==15))
# W4 DDR GSE17260 + PI3K null
ddr17260 = w4["panels"]["ddr"]["per_cohort"]["GSE17260"]
checks.append(("W4 DDR GSE17260 C=0.611", round(ddr17260["c_index"],3)==0.611))
checks.append(("W4 DDR GSE17260 HR=1.61", round(ddr17260["hr_per_1sd"],2)==1.61))
checks.append(("W4 DDR verdict CONSIDER", w4["panels"]["ddr"]["verdict"]=="CONSIDER"))
checks.append(("W4 PI3K 0 strict", w4["panels"]["pi3k_mtor"]["n_strict_signal"]==0))
checks.append(("W4 PI3K verdict discovery_only", w4["panels"]["pi3k_mtor"]["verdict"]=="discovery_only"))
pi3k_inv = w4["panels"]["pi3k_mtor"]["per_cohort"]["PMID17290060"]
checks.append(("W4 PI3K inverse C=0.383", round(pi3k_inv["c_index"],3)==0.383))
checks.append(("W4 both panels 17 cohorts", w4["panels"]["ddr"]["n_cohorts_adequate_power"]==17 and w4["panels"]["pi3k_mtor"]["n_cohorts_adequate_power"]==17))

allok = all(v for _,v in checks)
for name,v in checks:
    print(f"  [{'OK' if v else 'FAIL'}] {name}")
print("\nALL MANUSCRIPT HEADLINE NUMBERS MATCH RECEIPTS:", allok)


  [OK] W1 OOF C-index=0.539
  [OK] W1 OOF CI 0.500-0.579
  [OK] W1 OOF p=0.050
  [OK] W2 GSE9891 C=0.625
  [OK] W2 GSE9891 HR=1.36
  [OK] W2 GSE9891 PH p=0.045
  [OK] W2 verdict CONSIDER
  [OK] W2 15 indep cohorts
  [OK] W4 DDR GSE17260 C=0.611
  [OK] W4 DDR GSE17260 HR=1.61
  [OK] W4 DDR verdict CONSIDER
  [OK] W4 PI3K 0 strict
  [OK] W4 PI3K verdict discovery_only
  [OK] W4 PI3K inverse C=0.383
  [OK] W4 both panels 17 cohorts

ALL MANUSCRIPT HEADLINE NUMBERS MATCH RECEIPTS: True


Partitioning Riester test cohorts by potential compendium membership

In [53]:
import json

# Tested cohorts from W2 re-audit with their outcomes
w2 = json.load(open("/mnt/shared-workspace/shared/biomarker_reaudit/w2_prognostic_replication_reaudit.json"))
rc = w2["replication_cohorts"]

# GEO/dataset first-public years (verifiable from GEO series records / dataset publications).
# Riester 2014 used curatedOvarianData datasets available by ~2013. Anything first public <=2013
# COULD have been in the development compendium; later cohorts are unambiguously independent.
# Source years (GEO "Public on" / originating publication):
cohort_year = {
    "GSE9891": 2008,      # Tothill  -> IN Riester compendium (confirmed)
    "GSE26712": 2008,     # Bonome   -> in curatedOvarianData, likely compendium
    "GSE13876": 2009,     # Crijns   -> in curatedOvarianData, likely compendium
    "GSE18520": 2010,     # Mok      -> in curatedOvarianData, likely compendium
    "GSE30161": 2011,     # Ferriss  -> in curatedOvarianData, likely compendium
    "GSE14764": 2009,     # Denkert  -> in curatedOvarianData, likely compendium
    "E.MTAB.386": 2012,   # Bentink  -> in curatedOvarianData, likely compendium
    "GSE32062": 2012,     # Yoshihara (Japanese A) -> curatedOvarianData, likely compendium
    "GSE17260": 2010,     # Yoshihara (Japanese B) -> curatedOvarianData, likely compendium
    "GSE51088": 2013,     # Karlan   -> borderline; ~2013
    "GSE8842": 2008,      # Marchini/Crijns-adjacent -> older
    "GSE32063": 2012,     # Yoshihara small -> curated
    "GSE49997": 2013,     # Pils/Bentink-adjacent -> ~2013
    "PMID17290060": 2007, # Dressman -> curatedOvarianData, likely compendium
    "PMID19318476": 2009, # Denkert-adjacent -> older
}

print(f"{'cohort':<14}{'year':>6}{'n':>6}{'C-idx':>8}{'C_p':>10}{'HR_p':>10}  strict?  independence")
def strict(d):
    return (d.get("c_index_lower",0)>0.5) and ((d.get("hr_lower",0)>1.0) or (d.get("hr_upper",9)<1.0))

fully_indep = []
potential_compendium = []
for cid, d in rc.items():
    if "c_index" not in d: 
        continue
    yr = cohort_year.get(cid, None)
    indep = "INDEPENDENT" if (yr is not None and yr >= 2014) else "potential-compendium (<=2013)"
    s = "YES" if strict(d) else ""
    print(f"{cid:<14}{str(yr):>6}{d['n']:>6}{d['c_index']:>8.3f}{d['c_index_p']:>10.3g}{d['hr_p']:>10.3g}   {s:<6}  {indep}")
    if yr is not None and yr >= 2014:
        fully_indep.append(cid)
    else:
        potential_compendium.append(cid)

print("\nFully independent (post-Riester, >=2014):", fully_indep if fully_indep else "NONE")
print("Potential compendium (<=2013):", len(potential_compendium), "cohorts")
print("\nSTRICT-signal cohort GSE9891 year:", cohort_year["GSE9891"], "-> IN Riester development compendium (confirmed by Waldron/Riester/Birrer JNCI 2014)")


cohort          year     n   C-idx       C_p      HR_p  strict?  independence
E.MTAB.386      2012   129   0.527     0.517      0.33           potential-compendium (<=2013)
GSE13876        2009   157   0.557    0.0536     0.065           potential-compendium (<=2013)
GSE14764        2009    79   0.552     0.439     0.495           potential-compendium (<=2013)
GSE17260        2010   110   0.514     0.778     0.799           potential-compendium (<=2013)
GSE18520        2010    53   0.551     0.239    0.0843           potential-compendium (<=2013)
GSE26712        2008   185   0.554    0.0424    0.0877           potential-compendium (<=2013)
GSE30161        2011    58   0.531      0.55     0.829           potential-compendium (<=2013)
GSE32062        2012   257   0.509     0.767     0.758           potential-compendium (<=2013)
GSE32063        2012    40   0.508     0.924     0.926           potential-compendium (<=2013)
GSE49997        2013   194   0.465     0.426     0.685           po

Pulling Subject 260 PTEN + PI3K panel facts for reconciliation

In [55]:
import json
w5 = json.load(open("/mnt/shared-workspace/shared/biomarker_reaudit/w5_realdata_cin_reaudit.json"))
pten = w5["pten_nomination_verified"]
print("=== Subject 260 PTEN (structural, sWGS) ===")
for k,v in pten.items():
    print(f"  {k}: {v}")
print("\n=== PI3K/mTOR expression panel (W4, MetaGx bulk) ===")
w4 = json.load(open("/mnt/shared-workspace/shared/biomarker_reaudit/w4_baseline_scan_reaudit.json"))
p = w4["panels"]["pi3k_mtor"]
print("  n cohorts:", p["n_cohorts_adequate_power"], "| n strict:", p["n_strict_signal"], "| verdict:", p["verdict"])
# Is PTEN even in the expression panel? and how does PTEN expression behave?
print("\n  PI3K/mTOR panel genes include PTEN? -> panel = PIK3CA,PIK3CB,PIK3R1,PTEN,AKT1,AKT2,MTOR,RPTOR,RICTOR,TSC1,TSC2,STK11,INPP4B")
print("  Key point: panel score = mean-z of EXPRESSION across all 13 genes; a single-gene structural CN loss in 1 patient")
print("  is diluted 1/13 and is a DNA-level event, not necessarily reflected in bulk mRNA z-score.")


=== Subject 260 PTEN (structural, sWGS) ===
  dx_rel_cn: 0.9666
  relapse_rel_cn: 0.6915
  delta: -0.2751
  dx_call: neutral
  relapse_call: loss
  band_rule: loss<=0.8, neutral else
  acquired_transition: neutral->loss (VERIFIED)
  drives: PI3K/mTOR nomination at tier Consider (hard-capped below HIGH in prophet integration)

=== PI3K/mTOR expression panel (W4, MetaGx bulk) ===
  n cohorts: 17 | n strict: 0 | verdict: discovery_only

  PI3K/mTOR panel genes include PTEN? -> panel = PIK3CA,PIK3CB,PIK3R1,PTEN,AKT1,AKT2,MTOR,RPTOR,RICTOR,TSC1,TSC2,STK11,INPP4B
  Key point: panel score = mean-z of EXPRESSION across all 13 genes; a single-gene structural CN loss in 1 patient
  is diluted 1/13 and is a DNA-level event, not necessarily reflected in bulk mRNA z-score.


Building consensusOV re-execution + confound figure

In [57]:
import json, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
matplotlib.rcParams['font.family'] = ['Liberation Sans', 'Arimo', 'DejaVu Sans']
matplotlib.rcParams['svg.fonttype'] = 'none'
FIGDIR = "/mnt/results/biomarker_subsystem/figures"
PHY = dict(black="#000000", cream="#ECE9E2", offwhite="#FAF9F3", yellow="#E9ED4C",
           orange="#FF9400", green="#75A025", pink="#FD9BED", blue="#0279EE")

w3 = json.load(open("/mnt/shared-workspace/shared/biomarker_reaudit/w3_diagnostic_REEXEC.json"))
frac = w3["subtype_fractions_tcga"]  # dict subtype->frac
tumor = w3["composition_confound_tcga"]["percent_tumor_cells"]
gm = tumor["group_means"]  # subtype->mean tumor%
kp_t = w3["composition_confound_tcga"]["percent_tumor_cells"]["kruskal_p"]
kp_s = w3["composition_confound_tcga"]["percent_stromal_cells"]["kruskal_p"]

# consistent subtype order + short labels
order = ["DIF_consensus","IMR_consensus","MES_consensus","PRO_consensus"]
short = {"DIF_consensus":"DIF","IMR_consensus":"IMR","MES_consensus":"MES","PRO_consensus":"PRO"}
cols = {"DIF_consensus":PHY["blue"],"IMR_consensus":PHY["green"],
        "MES_consensus":PHY["orange"],"PRO_consensus":PHY["yellow"]}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 5.2))

# Panel A: subtype fractions
fr = [frac[o] for o in order]
bars = ax1.bar([short[o] for o in order], fr, color=[cols[o] for o in order],
               edgecolor=PHY["black"], linewidth=0.8)
for b, v in zip(bars, fr):
    ax1.text(b.get_x()+b.get_width()/2, v+0.006, f"{v:.3f}", ha="center", fontsize=9)
ax1.set_ylabel("Fraction of TCGA tumors (n=536)", fontsize=10.5)
ax1.set_title("A. consensusOV subtype composition (re-executed)", fontsize=11, loc="left")
ax1.set_ylim(0, 0.36)
ax1.spines[["top","right"]].set_visible(False)

# Panel B: mean tumor purity by subtype (the confound)
mv = [gm[o] for o in order]
bars2 = ax2.bar([short[o] for o in order], mv, color=[cols[o] for o in order],
                edgecolor=PHY["black"], linewidth=0.8)
for b, v in zip(bars2, mv):
    ax2.text(b.get_x()+b.get_width()/2, v+0.3, f"{v:.1f}", ha="center", fontsize=9)
grand = tumor["grand_mean"]
ax2.axhline(grand, color=PHY["black"], ls="--", lw=1.1)
ax2.text(3.4, grand+0.3, f"cohort mean {grand:.1f}", ha="right", fontsize=8.2, color=PHY["black"])
ax2.set_ylabel("Mean tumor purity (%)", fontsize=10.5)
ax2.set_ylim(70, 86)
ax2.set_title(f"B. Tumor-purity confound\nKruskal p={kp_t:.1e} (tumor), {kp_s:.1e} (stroma)", fontsize=11, loc="left")
ax2.spines[["top","right"]].set_visible(False)
# annotate MES as low outlier
mes_idx = order.index("MES_consensus")
ax2.annotate("MES = low-purity\noutlier", xy=(mes_idx, gm["MES_consensus"]),
             xytext=(mes_idx-0.15, 72.2), fontsize=8.4, color=PHY["orange"], fontweight="bold",
             arrowprops=dict(arrowstyle="->", color=PHY["orange"], lw=1.0))

fig.suptitle("Figure 3. consensusOV molecular subtype — full re-execution and composition confound (TCGA)",
             fontsize=11.8, x=0.02, ha="left")
plt.tight_layout(rect=[0,0,1,0.95])
png = f"{FIGDIR}/fig_v4_consensusov_reexec_confound.png"
svg = f"{FIGDIR}/fig_v4_consensusov_reexec_confound.svg"
fig.savefig(png, dpi=200, bbox_inches="tight")
fig.savefig(svg, bbox_inches="tight")
plt.close(fig)
print("saved:", png.split("/")[-1], "+ svg")
print("fractions:", {short[o]:round(frac[o],3) for o in order})
print("tumor% means:", {short[o]:gm[o] for o in order})


saved: fig_v4_consensusov_reexec_confound.png + svg
fractions: {'DIF': 0.297, 'IMR': 0.274, 'MES': 0.237, 'PRO': 0.192}
tumor% means: {'DIF': 82.45, 'IMR': 82.793, 'MES': 76.059, 'PRO': 81.426}


Fixing consensusOV figure annotation clipping/overlap

In [59]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 5.2))

# Panel A: subtype fractions
fr = [frac[o] for o in order]
bars = ax1.bar([short[o] for o in order], fr, color=[cols[o] for o in order],
               edgecolor=PHY["black"], linewidth=0.8)
for b, v in zip(bars, fr):
    ax1.text(b.get_x()+b.get_width()/2, v+0.006, f"{v:.3f}", ha="center", fontsize=9)
ax1.set_ylabel("Fraction of TCGA tumors (n=536)", fontsize=10.5)
ax1.set_title("A. consensusOV subtype composition (re-executed)", fontsize=11, loc="left")
ax1.set_ylim(0, 0.36)
ax1.spines[["top","right"]].set_visible(False)

# Panel B: mean tumor purity by subtype (the confound)
mv = [gm[o] for o in order]
bars2 = ax2.bar([short[o] for o in order], mv, color=[cols[o] for o in order],
                edgecolor=PHY["black"], linewidth=0.8)
for b, v in zip(bars2, mv):
    ax2.text(b.get_x()+b.get_width()/2, v+0.3, f"{v:.1f}", ha="center", fontsize=9)
grand = tumor["grand_mean"]
ax2.axhline(grand, color=PHY["black"], ls="--", lw=1.1)
# cohort-mean label placed top-LEFT, clear of bars
ax2.text(-0.35, grand+1.4, f"cohort mean {grand:.1f}", ha="left", fontsize=8.2, color=PHY["black"])
ax2.set_ylabel("Mean tumor purity (%)", fontsize=10.5)
ax2.set_ylim(70, 88)
ax2.set_xlim(-0.6, 3.6)
ax2.set_title(f"B. Tumor-purity confound\nKruskal p={kp_t:.1e} (tumor), {kp_s:.1e} (stroma)", fontsize=11, loc="left")
ax2.spines[["top","right"]].set_visible(False)
# annotate MES as low outlier - arrow from ABOVE the MES bar, text fully inside
mes_idx = order.index("MES_consensus")
ax2.annotate("MES = low-purity outlier", xy=(mes_idx, gm["MES_consensus"]+0.4),
             xytext=(mes_idx-0.05, 84.5), fontsize=8.6, color=PHY["orange"], fontweight="bold",
             ha="center", arrowprops=dict(arrowstyle="->", color=PHY["orange"], lw=1.1))

fig.suptitle("Figure 3. consensusOV molecular subtype — full re-execution and composition confound (TCGA)",
             fontsize=11.8, x=0.02, ha="left")
plt.tight_layout(rect=[0,0,1,0.95])
png = f"{FIGDIR}/fig_v4_consensusov_reexec_confound.png"
svg = f"{FIGDIR}/fig_v4_consensusov_reexec_confound.svg"
fig.savefig(png, dpi=200, bbox_inches="tight")
fig.savefig(svg, bbox_inches="tight")
plt.close(fig)
print("re-saved with fixed annotation placement")


re-saved with fixed annotation placement


Final placement fix for cohort-mean label

In [61]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 5.2))

fr = [frac[o] for o in order]
bars = ax1.bar([short[o] for o in order], fr, color=[cols[o] for o in order],
               edgecolor=PHY["black"], linewidth=0.8)
for b, v in zip(bars, fr):
    ax1.text(b.get_x()+b.get_width()/2, v+0.006, f"{v:.3f}", ha="center", fontsize=9)
ax1.set_ylabel("Fraction of TCGA tumors (n=536)", fontsize=10.5)
ax1.set_title("A. consensusOV subtype composition (re-executed)", fontsize=11, loc="left")
ax1.set_ylim(0, 0.36)
ax1.spines[["top","right"]].set_visible(False)

mv = [gm[o] for o in order]
bars2 = ax2.bar([short[o] for o in order], mv, color=[cols[o] for o in order],
                edgecolor=PHY["black"], linewidth=0.8)
for b, v in zip(bars2, mv):
    ax2.text(b.get_x()+b.get_width()/2, v+0.3, f"{v:.1f}", ha="center", fontsize=9)
grand = tumor["grand_mean"]
ax2.axhline(grand, color=PHY["black"], ls="--", lw=1.1)
# place cohort-mean label in clear gap between MES(2) and PRO(3), aligned just below the line
ax2.text(2.5, grand-1.3, f"cohort mean {grand:.1f}", ha="center", fontsize=8.2,
         color=PHY["black"], style="italic")
ax2.set_ylabel("Mean tumor purity (%)", fontsize=10.5)
ax2.set_ylim(70, 88)
ax2.set_xlim(-0.6, 3.6)
ax2.set_title(f"B. Tumor-purity confound\nKruskal p={kp_t:.1e} (tumor), {kp_s:.1e} (stroma)", fontsize=11, loc="left")
ax2.spines[["top","right"]].set_visible(False)
mes_idx = order.index("MES_consensus")
ax2.annotate("MES = low-purity outlier", xy=(mes_idx, gm["MES_consensus"]+0.4),
             xytext=(mes_idx-0.05, 84.8), fontsize=8.6, color=PHY["orange"], fontweight="bold",
             ha="center", arrowprops=dict(arrowstyle="->", color=PHY["orange"], lw=1.1))

fig.suptitle("Figure 3. consensusOV molecular subtype — full re-execution and composition confound (TCGA)",
             fontsize=11.8, x=0.02, ha="left")
plt.tight_layout(rect=[0,0,1,0.95])
png = f"{FIGDIR}/fig_v4_consensusov_reexec_confound.png"
svg = f"{FIGDIR}/fig_v4_consensusov_reexec_confound.svg"
fig.savefig(png, dpi=200, bbox_inches="tight")
fig.savefig(svg, bbox_inches="tight")
plt.close(fig)
print("re-saved: cohort-mean label moved to clear gap below line")


re-saved: cohort-mean label moved to clear gap below line


Build shared scaffold and sharded work manifest for FIFO demo

In [63]:
import os, csv, json, textwrap

# Shared scaffold for the distributed FIFO demonstration.
# NOTE: synthetic stand-in only. No EGA auth, no controlled data, no pasted credential used.
SHARED = "/mnt/shared-workspace/shared/sae_fifo_demo"
os.makedirs(SHARED, exist_ok=True)
os.makedirs(f"{SHARED}/csv_out", exist_ok=True)
os.makedirs(f"{SHARED}/logs", exist_ok=True)

# Build a synthetic "work manifest" mirroring the real ledger's SHAPE (25 pairs -> 50 files),
# but with synthetic file IDs. We deliberately do NOT copy real EGAF IDs here; this is a
# routing/throughput test, not a fetch of the real cohort.
N_PAIRS = 25
manifest = []
for p in range(1, N_PAIRS + 1):
    for side in ("dx", "relapse"):
        manifest.append({
            "work_id": f"SYNTH_{p:03d}_{side}",
            "subject": p,
            "side": side,
            # synthetic per-file "size" and seed so the fake reads are deterministic
            "seed": p * 10 + (0 if side == "dx" else 1),
        })

with open(f"{SHARED}/work_manifest.csv", "w", newline="") as fh:
    w = csv.DictWriter(fh, fieldnames=["work_id", "subject", "side", "seed", "assigned_worker"])
    w.writeheader()
    for i, row in enumerate(manifest):
        row["assigned_worker"] = f"worker-{i % 5}"   # round-robin shard across 5 workers
        w.writerow(row)

# summarize the shard
from collections import Counter
shards = Counter(f"worker-{i % 5}" for i in range(len(manifest)))
print(f"Synthetic work manifest: {len(manifest)} files ({N_PAIRS} pairs x 2 sides)")
print("Shard distribution (round-robin modulo 5):")
for wkr in sorted(shards):
    print(f"  {wkr}: {shards[wkr]} files")
print(f"\nScaffold at: {SHARED}")
print("  work_manifest.csv, csv_out/, logs/  (shared across all workers)")


Synthetic work manifest: 50 files (25 pairs x 2 sides)
Shard distribution (round-robin modulo 5):
  worker-0: 10 files
  worker-1: 10 files
  worker-2: 10 files
  worker-3: 10 files
  worker-4: 10 files

Scaffold at: /mnt/shared-workspace/shared/sae_fifo_demo
  work_manifest.csv, csv_out/, logs/  (shared across all workers)


Reassemble per-file CSVs into breakpoints table and verify counts

In [65]:
import glob, csv, os
from collections import Counter

SHARED = "/mnt/shared-workspace/shared/sae_fifo_demo"
csvs = sorted(glob.glob(f"{SHARED}/csv_out/*.csv"))
logs = sorted(glob.glob(f"{SHARED}/logs/*.log"))

rows = []
for c in csvs:
    with open(c) as fh:
        rows.extend(list(csv.DictReader(fh)))

by_worker = Counter(r["worker"] for r in rows)
subjects = sorted({int(r["subject"]) for r in rows})
sides = Counter(r["side"] for r in rows)

# reassemble into pairs and compute a demo breakpoint_delta
pairs = {}
for r in rows:
    pairs.setdefault(int(r["subject"]), {})[r["side"]] = int(r["breakpoints"])
complete_pairs = {s: v for s, v in pairs.items() if "dx" in v and "relapse" in v}

# write reassembled table to shared + results
recon = f"{SHARED}/breakpoints_reassembled.csv"
with open(recon, "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["subject", "dx_breakpoints", "relapse_breakpoints", "breakpoint_delta"])
    for s in sorted(complete_pairs):
        dx = complete_pairs[s]["dx"]; rl = complete_pairs[s]["relapse"]
        w.writerow([s, dx, rl, rl - dx])

print("=== DISTRIBUTED FIFO DEMO — REASSEMBLY ===")
print(f"per-file CSVs collected from shared store : {len(csvs)}  (expected 50)")
print(f"worker log files                          : {len(logs)}  ({', '.join(os.path.basename(l) for l in logs)})")
print(f"rows total                                : {len(rows)}")
print(f"files per worker                          : {dict(sorted(by_worker.items()))}")
print(f"unique subjects                           : {len(subjects)}  (expected 25) -> {subjects[0]}..{subjects[-1]}")
print(f"sides                                     : {dict(sides)}  (expected dx=25, relapse=25)")
print(f"complete dx+relapse pairs reassembled     : {len(complete_pairs)}  (expected 25)")
print(f"\nReassembled table written: breakpoints_reassembled.csv")


Execution interrupted by user pause. The Python kernel is still alive with all variables and state preserved.

In [66]:
import os, csv, hashlib
from datetime import datetime

RDV = "/mnt/results/biomarker_subsystem/real_data_validation"
core_files = [
    "breakpoint_architecture_with_subj260.csv",
    "breakpoint_architecture_pilot.csv",
    "breakpoint_delta_by_chrom.csv",
    "subject_ledger.csv",
]

def provenance(path):
    sz = os.path.getsize(path)
    mt = datetime.fromtimestamp(os.path.getmtime(path)).isoformat()
    with open(path, "rb") as f:
        h = hashlib.md5(f.read()).hexdigest()[:12]
    with open(path) as f:
        nl = sum(1 for _ in f)
    with open(path) as f:
        cols = next(csv.reader(f))
    return {"size": sz, "mtime": mt, "md5": h, "rows": nl, "columns": cols}

print("=== PROVENANCE HEADERS ===")
prov = {}
for fn in core_files:
    p = os.path.join(RDV, fn)
    if not os.path.exists(p):
        raise FileNotFoundError(f"MISSING required file: {p}")
    prov[fn] = provenance(p)
    print(f"{fn}")
    for k, v in prov[fn].items():
        print(f"  {k}: {v}")
print("\nAll required files present and intact.")


=== PROVENANCE HEADERS ===


breakpoint_architecture_with_subj260.csv
  size: 563
  mtime: 2026-07-21T06:28:09
  md5: 6ad5d3ff52cb
  rows: 6
  columns: ['subject', 'dx_sample', 'relapse_sample', 'dx_breakpoints', 'relapse_breakpoints', 'breakpoint_delta', 'net_gained_breakpoints', 'n_chromosomes_gained', 'top_focus_chrom', 'top_focus_fraction', 'top_chrom_cn_levels', 'architecture_signature']


breakpoint_architecture_pilot.csv
  size: 489
  mtime: 2026-07-21T05:31:56
  md5: 1a9f0e980f7d
  rows: 5
  columns: ['subject', 'dx_sample', 'relapse_sample', 'dx_breakpoints', 'relapse_breakpoints', 'breakpoint_delta', 'net_gained_breakpoints', 'n_chromosomes_gained', 'top_focus_chrom', 'top_focus_fraction', 'top_chrom_cn_levels', 'architecture_signature']


breakpoint_delta_by_chrom.csv
  size: 660
  mtime: 2026-07-21T05:31:57
  md5: 06d08ea6aa70
  rows: 75
  columns: ['subject', 'chrom', 'breakpoint_delta']


subject_ledger.csv
  size: 31843
  mtime: 2026-07-21T06:28:13
  md5: 52d139ac0286
  rows: 183
  columns: ['subject', 'dx_sample', 'dx_egaf', 'dx_egan', 'dx_site', 'relapse_sample', 'relapse_egaf', 'relapse_egan', 'relapse_site', 'histology', 'processed', 'dx_breakpoints', 'relapse_breakpoints', 'breakpoint_delta', 'architecture_signature', 'n_chromosomes_gained', 'cn_status', 'high_breakpoint']

All required files present and intact.


In [67]:
import csv
RDV = "/mnt/results/biomarker_subsystem/real_data_validation"

# Load architecture summary (5 computed pairs)
arch = list(csv.DictReader(open(f"{RDV}/breakpoint_architecture_with_subj260.csv")))

print("=== RECOMPUTING BREAKPOINT DELTAS FROM ARCHITECTURE FILE ===")
print(f"{'subject':>6} {'dx_sample':>14} {'relapse_sample':>14} {'dx_bp':>6} {'rel_bp':>7} {'stored_delta':>12} {'recomputed_delta':>16} {'match':>5}")
for r in arch:
    dx = int(r['dx_breakpoints'])
    rel = int(r['relapse_breakpoints'])
    stored = int(r['breakpoint_delta'])
    recomp = rel - dx
    match = "OK" if recomp == stored else "FAIL"
    print(f"{r['subject']:>6} {r['dx_sample']:>14} {r['relapse_sample']:>14} {dx:>6} {rel:>7} {stored:>12} {recomp:>16} {match:>5}")

print("\n=== RANKING 5 COMPUTED PAIRS BY SIGNAL STRENGTH (breakpoint_delta) ===")
ranked = sorted(arch, key=lambda r: -int(r['breakpoint_delta']))
print(f"{'rank':>4} {'subject':>7} {'dx->relapse':>22} {'delta':>6} {'signature':>26}")
for i, r in enumerate(ranked, 1):
    pair = f"{r['dx_sample']}->{r['relapse_sample']}"
    print(f"{i:>4} {r['subject']:>7} {pair:>22} {r['breakpoint_delta']:>6} {r['architecture_signature']:>26}")

# Cross-check against subject_ledger.csv for the 5 computed subjects
ledger = {r['subject']: r for r in csv.DictReader(open(f"{RDV}/subject_ledger.csv"))}
print("\n=== CROSS-CHECK: architecture file vs subject_ledger.csv ===")
print(f"{'subject':>7} {'arch_delta':>10} {'ledger_delta':>12} {'arch_sig':>26} {'ledger_sig':>26}")
for r in arch:
    subj = r['subject']
    l = ledger.get(subj, {})
    print(f"{subj:>7} {r['breakpoint_delta']:>10} {l.get('breakpoint_delta','NA'):>12} {r['architecture_signature']:>26} {l.get('architecture_signature','NA'):>26}")


=== RECOMPUTING BREAKPOINT DELTAS FROM ARCHITECTURE FILE ===
subject      dx_sample relapse_sample  dx_bp  rel_bp stored_delta recomputed_delta match
   191         IM_280     JBLAB-4944    127     380          253              253    OK
     6         IM_274     JBLAB-4110     17     118          101              101    OK
   260         IM_426    JBLAB-19323     75     135           60               60    OK
   119         IM_249     JBLAB-4232    148     166           18               18    OK
   109         IM_373     JBLAB-4225     59      52           -7               -7    OK

=== RANKING 5 COMPUTED PAIRS BY SIGNAL STRENGTH (breakpoint_delta) ===
rank subject            dx->relapse  delta                  signature
   1     191     IM_280->JBLAB-4944    253   dispersed_stochastic_CIN
   2       6     IM_274->JBLAB-4110    101   dispersed_stochastic_CIN
   3     260    IM_426->JBLAB-19323     60   dispersed_stochastic_CIN
   4     119     IM_249->JBLAB-4232     18 intermediate_re

In [68]:
import csv
from collections import defaultdict
RDV = "/mnt/results/biomarker_subsystem/real_data_validation"

rows = list(csv.DictReader(open(f"{RDV}/breakpoint_delta_by_chrom.csv")))
by_subject = defaultdict(list)
for r in rows:
    by_subject[r['subject']].append((r['chrom'], int(r['breakpoint_delta'])))

print("=== PER-CHROMOSOME BREAKPOINT DELTA: 191 vs 109 ===")
for subj in ['191', '109']:
    chroms = by_subject[subj]
    total = sum(d for _, d in chroms)
    gains = [(c, d) for c, d in chroms if d > 0]
    losses = [(c, d) for c, d in chroms if d < 0]
    top = max(chroms, key=lambda x: x[1])
    top_frac = top[1] / total if total else 0
    print(f"\nSubject {subj}:")
    print(f"  total new breakpoints (sum per-chrom): {total}")
    print(f"  chromosomes with net gain: {len(gains)} ({[c for c,_ in gains]})")
    print(f"  chromosomes with net loss: {len(losses)} ({[c for c,_ in losses]})")
    print(f"  top chromosome: {top[0]} with {top[1]} new breakpoints ({top_frac:.1%} of total)")
    print(f"  per-chrom distribution: {dict(chroms)}")

# Sum-to-total check against architecture file
arch = {r['subject']: r for r in csv.DictReader(open(f"{RDV}/breakpoint_architecture_with_subj260.csv"))}
print("\n=== SUM-TO-TOTAL CHECK ===")
for subj in ['191', '109']:
    per_chrom_total = sum(d for _, d in by_subject[subj])
    arch_delta = int(arch[subj]['breakpoint_delta'])
    match = "OK" if per_chrom_total == arch_delta else "FAIL"
    print(f"  {subj}: per-chrom sum={per_chrom_total}, architecture delta={arch_delta}, {match}")


=== PER-CHROMOSOME BREAKPOINT DELTA: 191 vs 109 ===

Subject 191:
  total new breakpoints (sum per-chrom): 253
  chromosomes with net gain: 21 (['1', '2', '3', '4', '5', '6', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22'])
  chromosomes with net loss: 0 ([])
  top chromosome: 1 with 29 new breakpoints (11.5% of total)
  per-chrom distribution: {'1': 29, '2': 19, '3': 22, '4': 7, '5': 14, '6': 4, '8': 22, '9': 14, '10': 7, '11': 17, '12': 13, '13': 1, '14': 19, '15': 6, '16': 5, '17': 7, '18': 7, '19': 4, '20': 10, '21': 7, '22': 19}

Subject 109:
  total new breakpoints (sum per-chrom): -7
  chromosomes with net gain: 3 (['9', '14', '20'])
  chromosomes with net loss: 9 (['2', '4', '6', '7', '11', '12', '13', '17', '19'])
  top chromosome: 14 with 4 new breakpoints (-57.1% of total)
  per-chrom distribution: {'2': -2, '4': -1, '6': -1, '7': -2, '9': 2, '11': -3, '12': -2, '13': -1, '14': 4, '17': -1, '19': -1, '20': 1}

=== SUM-TO-TOTAL CHECK ==

In [69]:
import csv
RDV = "/mnt/results/biomarker_subsystem/real_data_validation"

arch = list(csv.DictReader(open(f"{RDV}/breakpoint_architecture_with_subj260.csv")))

# Provisional cutoffs as documented in the provenance record
CUTOFFS = {
    'dispersed_stochastic_CIN': (50, float('inf')),
    'intermediate_rearrangement': (10, 49),
    'modest_gain': (float('-inf'), 9),
}

def expected_label(delta):
    d = int(delta)
    if d >= 50:
        return 'dispersed_stochastic_CIN'
    elif d >= 10:
        return 'intermediate_rearrangement'
    else:
        return 'modest_gain'

print("=== VERIFYING ARCHITECTURE-SIGNATURE CUTOFFS ===")
print("Provisional cutoffs (n=5, under-determined):")
for label, (lo, hi) in CUTOFFS.items():
    hi_str = str(hi) if hi != float('inf') else 'inf'
    print(f"  {label}: delta in [{lo}, {hi_str})")
print()
print(f"{'subject':>7} {'delta':>6} {'stored_label':>26} {'expected_label':>26} {'match':>5}")
all_ok = True
for r in arch:
    exp = expected_label(r['breakpoint_delta'])
    stored = r['architecture_signature']
    match = "OK" if exp == stored else "FAIL"
    if match == "FAIL":
        all_ok = False
    print(f"{r['subject']:>7} {r['breakpoint_delta']:>6} {stored:>26} {exp:>26} {match:>5}")
print(f"\nAll labels follow cutoffs: {all_ok}")


=== VERIFYING ARCHITECTURE-SIGNATURE CUTOFFS ===
Provisional cutoffs (n=5, under-determined):
  dispersed_stochastic_CIN: delta in [50, inf)
  intermediate_rearrangement: delta in [10, 49)
  modest_gain: delta in [-inf, 9)

subject  delta               stored_label             expected_label match
    191    253   dispersed_stochastic_CIN   dispersed_stochastic_CIN    OK
      6    101   dispersed_stochastic_CIN   dispersed_stochastic_CIN    OK
    260     60   dispersed_stochastic_CIN   dispersed_stochastic_CIN    OK
    119     18 intermediate_rearrangement intermediate_rearrangement    OK
    109     -7                modest_gain                modest_gain    OK

All labels follow cutoffs: True


In [70]:
import csv, json
RDV = "/mnt/results/biomarker_subsystem/real_data_validation"

# Load everything needed for the final report
arch = list(csv.DictReader(open(f"{RDV}/breakpoint_architecture_with_subj260.csv")))
arch_by_subj = {r['subject']: r for r in arch}
chrom_rows = list(csv.DictReader(open(f"{RDV}/breakpoint_delta_by_chrom.csv")))
ledger = {r['subject']: r for r in csv.DictReader(open(f"{RDV}/subject_ledger.csv"))}

# Recompute 191 vs 109 contrast from derived files
from collections import defaultdict
by_subject = defaultdict(list)
for r in chrom_rows:
    by_subject[r['subject']].append((r['chrom'], int(r['breakpoint_delta'])))

def dispersion_stats(subj):
    chroms = by_subject[subj]
    total = sum(d for _, d in chroms)
    gains = [(c, d) for c, d in chroms if d > 0]
    losses = [(c, d) for c, d in chroms if d < 0]
    top = max(chroms, key=lambda x: x[1])
    top_frac = top[1] / total if total else 0
    return {
        'total': total,
        'n_gains': len(gains),
        'n_losses': len(losses),
        'top_chrom': top[0],
        'top_count': top[1],
        'top_frac': top_frac,
    }

s191 = dispersion_stats('191')
s109 = dispersion_stats('109')

# Transcript-only values (cannot be recomputed from surviving files)
quarantine = [
    ("Subject 191 frac_genome_altered_gt0.3log2", "0.5858 (58.6%)", "transcript i:15523"),
    ("Subject 109 frac_genome_altered_gt0.3log2", "0.1210 (12.1%)", "transcript i:15523"),
    ("Subject 191 net_log2_shift", "+0.2780", "transcript i:15523"),
    ("Subject 109 net_log2_shift", "-0.0003", "transcript i:15523"),
    ("Subject 191 max_gain_log2", "2.142", "transcript i:15523"),
    ("Subject 109 max_gain_log2", "1.319", "transcript i:15523"),
    ("Subject 191 max_loss_log2", "-1.006", "transcript i:15523"),
    ("Subject 109 max_loss_log2", "-1.310", "transcript i:15523"),
    ("Sig7 bin-level n_breakpoints IM_280", "120", "transcript i:15525"),
    ("Ledger architecture dx_breakpoints IM_280", "127", "breakpoint_architecture_with_subj260.csv"),
    ("Sig7 bin-level n_breakpoints JBLAB-4944", "386", "transcript i:15525"),
    ("Ledger architecture relapse_breakpoints JBLAB-4944", "380", "breakpoint_architecture_with_subj260.csv"),
]

print("=== AUDIT SUMMARY PREVIEW ===")
print(f"Subject 191: delta={arch_by_subj['191']['breakpoint_delta']} | top chrom {s191['top_chrom']} = {s191['top_count']} ({s191['top_frac']:.1%}) | {s191['n_gains']}/22 chromosomes gained")
print(f"Subject 109: delta={arch_by_subj['109']['breakpoint_delta']} | top chrom {s109['top_chrom']} = {s109['top_count']} ({s109['top_frac']:.1%}) | {s109['n_gains']} gained, {s109['n_losses']} lost")
print(f"\nQuarantined transcript-only values: {len(quarantine)}")
for label, value, source in quarantine:
    print(f"  {label}: {value} [{source}]")

# Save intermediate for markdown generation
import pickle
with open('/workspace/audit_intermediate.pkl', 'wb') as f:
    pickle.dump({'arch': arch, 'arch_by_subj': arch_by_subj, 's191': s191, 's109': s109, 'quarantine': quarantine}, f)
print("\nIntermediate saved to /workspace/audit_intermediate.pkl")


=== AUDIT SUMMARY PREVIEW ===
Subject 191: delta=253 | top chrom 1 = 29 (11.5%) | 21/22 chromosomes gained
Subject 109: delta=-7 | top chrom 14 = 4 (-57.1%) | 3 gained, 9 lost

Quarantined transcript-only values: 12
  Subject 191 frac_genome_altered_gt0.3log2: 0.5858 (58.6%) [transcript i:15523]
  Subject 109 frac_genome_altered_gt0.3log2: 0.1210 (12.1%) [transcript i:15523]
  Subject 191 net_log2_shift: +0.2780 [transcript i:15523]
  Subject 109 net_log2_shift: -0.0003 [transcript i:15523]
  Subject 191 max_gain_log2: 2.142 [transcript i:15523]
  Subject 109 max_gain_log2: 1.319 [transcript i:15523]
  Subject 191 max_loss_log2: -1.006 [transcript i:15523]
  Subject 109 max_loss_log2: -1.310 [transcript i:15523]
  Sig7 bin-level n_breakpoints IM_280: 120 [transcript i:15525]
  Ledger architecture dx_breakpoints IM_280: 127 [breakpoint_architecture_with_subj260.csv]
  Sig7 bin-level n_breakpoints JBLAB-4944: 386 [transcript i:15525]
  Ledger architecture relapse_breakpoints JBLAB-4944: 

In [71]:
import pickle, csv
from collections import defaultdict
RDV = "/mnt/results/biomarker_subsystem/real_data_validation"

# Load intermediate
with open('/workspace/audit_intermediate.pkl', 'rb') as f:
    inter = pickle.load(f)
arch, arch_by_subj, s191, s109, quarantine = inter['arch'], inter['arch_by_subj'], inter['s191'], inter['s109'], inter['quarantine']

# Load per-chrom data for figure
chrom_rows = list(csv.DictReader(open(f"{RDV}/breakpoint_delta_by_chrom.csv")))
by_subject = defaultdict(list)
for r in chrom_rows:
    by_subject[r['subject']].append((r['chrom'], int(r['breakpoint_delta'])))

# Build markdown report
report = f"""# Subject 191 Copy-Number Signal Audit

**Scope:** Re-audit of the 5 computed BriTROC-1 serial copy-number pairs using ONLY
derived artifacts surviving on disk. No new EGA connection, no new credentials, no
new BAM downloads. Every number below is tagged with its source.

**Key limitation:** The raw BAMs and the per-bin/segment CN matrices
(`pilot_bin_relative_cn.csv`, `pilot_segmented_cn.csv`) were written to ephemeral
`/workspace/cn_out` on 2026-07-20 and no longer exist. Therefore breakpoints cannot
be re-derived from sequence or copy-number values; this audit recomputes and cross-
checks the *derived summary layer* only.

---

## 1. Provenance of source files (printed in-run)

| file | size | rows | md5 (first 12) | mtime |
|---|---|---|---|---|
| breakpoint_architecture_with_subj260.csv | 563 B | 6 | 6ad5d3ff52cb | 2026-07-21 06:28:09 |
| breakpoint_architecture_pilot.csv | 489 B | 5 | 1a9f0e980f7d | 2026-07-21 05:31:56 |
| breakpoint_delta_by_chrom.csv | 660 B | 75 | 06d08ea6aa70 | 2026-07-21 05:31:57 |
| subject_ledger.csv | 31,843 B | 183 | 52d139ac0286 | 2026-07-21 06:28:13 |

---

## 2. Recomputed breakpoint deltas and signal ranking

Delta recomputed as `relapse_breakpoints - dx_breakpoints` from
`breakpoint_architecture_with_subj260.csv`.

| rank | subject | dx → relapse | dx_bp | relapse_bp | delta | signature |
|---|---|---|---|---|---|---|
| 1 | 191 | IM_280 → JBLAB-4944 | 127 | 380 | **253** | dispersed_stochastic_CIN |
| 2 | 6 | IM_274 → JBLAB-4110 | 17 | 118 | 101 | dispersed_stochastic_CIN |
| 3 | 260 | IM_426 → JBLAB-19323 | 75 | 135 | 60 | dispersed_stochastic_CIN |
| 4 | 119 | IM_249 → JBLAB-4232 | 148 | 166 | 18 | intermediate_rearrangement |
| 5 | 109 | IM_373 → JBLAB-4225 | 59 | 52 | −7 | modest_gain |

**Cross-check:** recomputed delta equals stored delta for all 5 pairs.
`architecture_with_subj260.csv` and `subject_ledger.csv` agree on delta and signature
for all 5 pairs.

---

## 3. Subject 191 vs Subject 109: per-chromosome dispersion (recomputed)

From `breakpoint_delta_by_chrom.csv`.

### Subject 191 (strongest signal)
- Total new breakpoints: **253** [recomputed-from: breakpoint_delta_by_chrom.csv]
- Chromosomes with net breakpoint gain: **21 of 22** [recomputed-from: breakpoint_delta_by_chrom.csv]
- Top chromosome: **chr1**, +29 breakpoints = **11.5%** of total new breakpoints [recomputed-from: breakpoint_delta_by_chrom.csv]
- Interpretation: the gain is **genome-wide / dispersed**, not a focal catastrophe. No single chromosome dominates.

### Subject 109 (near-clonal/stable)
- Total breakpoint delta: **−7** [recomputed-from: breakpoint_delta_by_chrom.csv]
- Chromosomes with net gain: **3** (chr9, chr14, chr20) [recomputed-from: breakpoint_delta_by_chrom.csv]
- Chromosomes with net loss: **9** [recomputed-from: breakpoint_delta_by_chrom.csv]
- Top chromosome: **chr14**, +4 breakpoints [recomputed-from: breakpoint_delta_by_chrom.csv]
- Interpretation: near-neutral, balanced small gains/losses; no CIN burst.

**Sum-to-total check:** per-chromosome sums equal the architecture-file deltas for
both 191 (253 = 253) and 109 (−7 = −7).

---

## 4. Architecture-signature cutoffs (verified)

Provisional cutoffs (n=5, under-determined):
- delta ≥ 50 → dispersed_stochastic_CIN
- 10 ≤ delta ≤ 49 → intermediate_rearrangement
- delta < 10 → modest_gain

All 5 stored labels follow these cutoffs.

---

## 5. Transcript-only values (NOT recomputable from surviving files)

These values come from the 2026-07-20 execution transcript and are quarantined here
because the inputs needed to recompute them (per-bin CN matrix, raw BAMs) are gone.

| quantity | value | source |
|---|---|---|
| Subject 191 frac_genome_altered_gt0.3log2 | 0.5858 (58.6%) | transcript i:15523 |
| Subject 109 frac_genome_altered_gt0.3log2 | 0.1210 (12.1%) | transcript i:15523 |
| Subject 191 net_log2_shift | +0.2780 | transcript i:15523 |
| Subject 109 net_log2_shift | −0.0003 | transcript i:15523 |
| Subject 191 max_gain_log2 | 2.142 | transcript i:15523 |
| Subject 109 max_gain_log2 | 1.319 | transcript i:15523 |
| Subject 191 max_loss_log2 | −1.006 | transcript i:15523 |
| Subject 109 max_loss_log2 | −1.310 | transcript i:15523 |

### Known dual-definition discrepancy (also transcript-only vs derived-file)
| sample | Sig7 bin-level n_breakpoints | Ledger architecture breakpoints | difference |
|---|---|---|---|
| IM_280 (191 dx) | 120 [transcript i:15525] | 127 [breakpoint_architecture_with_subj260.csv] | −7 |
| JBLAB-4944 (191 relapse) | 386 [transcript i:15525] | 380 [breakpoint_architecture_with_subj260.csv] | +6 |

These are two different segment-boundary definitions. The headline "127 → 380, +253"
uses the ledger architecture definition.

---

## 6. Bottom line

- The strongest signal is **Subject 191**, with **+253 new breakpoints** spanning
  **21 of 22 autosomes**, top chromosome contributing only **11.5%**.
- This pattern is **dispersed stochastic CIN**, not a focal "master switch"
  catastrophe.
- **Subject 109** is the contrast: near-clonal/stable, delta **−7**, small
  gains/losses balanced across 12 chromosomes.
- The 58.6% genome-altered figure for 191 and 12.1% for 109 are receipt-cited from
  the transcript, not recomputable now; the breakpoint-dispersion conclusion is
  recomputable from the surviving derived files.
- n=5; architecture-signature cutoffs are provisional and under-determined.
"""

out_path = f"{RDV}/SUBJECT_191_AUDIT.md"
with open(out_path, 'w') as f:
    f.write(report)
print(f"Wrote {out_path} ({len(report)} chars)")


Wrote /mnt/results/biomarker_subsystem/real_data_validation/SUBJECT_191_AUDIT.md (5394 chars)


In [72]:
import matplotlib
matplotlib.rcParams['font.family'] = ['Liberation Sans', 'Arimo', 'DejaVu Sans']
matplotlib.rcParams['svg.fonttype'] = 'none'

import matplotlib.pyplot as plt
import seaborn as sns
import csv
from collections import defaultdict

RDV = "/mnt/results/biomarker_subsystem/real_data_validation"

# Load per-chrom data
chrom_rows = list(csv.DictReader(open(f"{RDV}/breakpoint_delta_by_chrom.csv")))
by_subject = defaultdict(list)
for r in chrom_rows:
    by_subject[r['subject']].append((r['chrom'], int(r['breakpoint_delta'])))

# Phylo palette (black, off-white, cream, lime, orange, green, pink, blue)
phylo = ['#000000', '#ECE9E2', '#FAF9F3', '#E9ED4C', '#FF9400', '#75A025', '#FD9BED', '#0279EE']

fig, axes = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
for ax, subj, title, color in zip(axes, ['191', '109'],
                                   ['Subject 191: IM_280 → JBLAB-4944 (delta=+253)',
                                    'Subject 109: IM_373 → JBLAB-4225 (delta=−7)'],
                                   [phylo[3], phylo[4]]):
    chroms, deltas = zip(*sorted(by_subject[subj], key=lambda x: int(x[0])))
    x = list(range(len(chroms)))
    ax.bar(x, deltas, color=color, edgecolor='black', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(chroms)
    ax.set_ylabel('breakpoint delta (relapse − dx)')
    ax.set_title(title, loc='left')
    # annotate top gain/loss
    top_idx = max(range(len(deltas)), key=lambda i: deltas[i])
    bot_idx = min(range(len(deltas)), key=lambda i: deltas[i])
    ax.annotate(f'{deltas[top_idx]}', xy=(x[top_idx], deltas[top_idx]),
                xytext=(0, 5), textcoords='offset points', ha='center', fontsize=8)
    ax.annotate(f'{deltas[bot_idx]}', xy=(x[bot_idx], deltas[bot_idx]),
                xytext=(0, -10), textcoords='offset points', ha='center', fontsize=8)

axes[-1].set_xlabel('chromosome')
plt.tight_layout()

png_path = f"{RDV}/fig_subject191_vs_109_per_chrom.png"
svg_path = f"{RDV}/fig_subject191_vs_109_per_chrom.svg"
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.savefig(svg_path, bbox_inches='tight')
plt.close()

print(f"Saved {png_path}")
print(f"Saved {svg_path}")


Saved /mnt/results/biomarker_subsystem/real_data_validation/fig_subject191_vs_109_per_chrom.png
Saved /mnt/results/biomarker_subsystem/real_data_validation/fig_subject191_vs_109_per_chrom.svg


In [73]:
import matplotlib
matplotlib.rcParams['font.family'] = ['Liberation Sans', 'Arimo', 'DejaVu Sans']
matplotlib.rcParams['svg.fonttype'] = 'none'

import matplotlib.pyplot as plt
import csv
from collections import defaultdict

RDV = "/mnt/results/biomarker_subsystem/real_data_validation"

chrom_rows = list(csv.DictReader(open(f"{RDV}/breakpoint_delta_by_chrom.csv")))
by_subject = defaultdict(dict)
for r in chrom_rows:
    by_subject[r['subject']][r['chrom']] = int(r['breakpoint_delta'])

phylo = ['#000000', '#ECE9E2', '#FAF9F3', '#E9ED4C', '#FF9400', '#75A025', '#FD9BED', '#0279EE']

chrom_order = [str(i) for i in range(1, 23)]

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
for ax, subj, title, color in zip(axes, ['191', '109'],
                                   ['Subject 191: IM_280 → JBLAB-4944 (delta=+253)',
                                    'Subject 109: IM_373 → JBLAB-4225 (delta=−7)'],
                                   [phylo[3], phylo[4]]):
    deltas = [by_subject[subj].get(c, 0) for c in chrom_order]
    x = list(range(len(chrom_order)))
    bars = ax.bar(x, deltas, color=color, edgecolor='black', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(chrom_order)
    ax.set_ylabel('breakpoint delta (relapse − dx)')
    ax.set_title(title, loc='left')
    # annotate max and min
    max_idx = max(range(len(deltas)), key=lambda i: deltas[i])
    min_idx = min(range(len(deltas)), key=lambda i: deltas[i])
    ax.annotate(f'{deltas[max_idx]}', xy=(x[max_idx], deltas[max_idx]),
                xytext=(0, 5), textcoords='offset points', ha='center', fontsize=8)
    ax.annotate(f'{deltas[min_idx]}', xy=(x[min_idx], deltas[min_idx]),
                xytext=(0, -10), textcoords='offset points', ha='center', fontsize=8)

axes[-1].set_xlabel('chromosome')
plt.tight_layout()

png_path = f"{RDV}/fig_subject191_vs_109_per_chrom.png"
svg_path = f"{RDV}/fig_subject191_vs_109_per_chrom.svg"
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.savefig(svg_path, bbox_inches='tight')
plt.close()

print(f"Regenerated {png_path}")
print(f"Regenerated {svg_path}")


Regenerated /mnt/results/biomarker_subsystem/real_data_validation/fig_subject191_vs_109_per_chrom.png
Regenerated /mnt/results/biomarker_subsystem/real_data_validation/fig_subject191_vs_109_per_chrom.svg


In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = ['Liberation Sans', 'Arimo', 'DejaVu Sans']
matplotlib.rcParams['svg.fonttype'] = 'none'

import matplotlib.pyplot as plt
import csv
from collections import defaultdict

RDV = "/mnt/results/biomarker_subsystem/real_data_validation"

chrom_rows = list(csv.DictReader(open(f"{RDV}/breakpoint_delta_by_chrom.csv")))
by_subject = defaultdict(dict)
for r in chrom_rows:
    by_subject[r['subject']][r['chrom']] = int(r['breakpoint_delta'])

phylo = ['#000000', '#ECE9E2', '#FAF9F3', '#E9ED4C', '#FF9400', '#75A025', '#FD9BED', '#0279EE']
chrom_order = [str(i) for i in range(1, 23)]

fig, axes = plt.subplots(2, 1, figsize=(12, 9))
for ax, subj, title, color in zip(axes, ['191', '109'],
                                   ['Subject 191: IM_280 → JBLAB-4944 (delta=+253)',
                                    'Subject 109: IM_373 → JBLAB-4225 (delta=−7)'],
                                   [phylo[3], phylo[4]]):
    deltas = [by_subject[subj].get(c, 0) for c in chrom_order]
    x = list(range(len(chrom_order)))
    # Render zero-height bars as thin horizontal lines so every chromosome is visible
    bars = []
    for xi, d in zip(x, deltas):
        if d == 0:
            bar = ax.bar(xi, 0.15, bottom=-0.075, color='white', edgecolor='black', linewidth=0.5)
        else:
            bar = ax.bar(xi, d, color=color, edgecolor='black', linewidth=0.5)
        bars.append(bar)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(chrom_order)
    ax.set_ylabel('breakpoint delta (relapse − dx)')
    ax.set_xlabel('chromosome')
    ax.set_title(title, loc='left')
    # annotate max and min (excluding zeros)
    nonzero = [(i, d) for i, d in enumerate(deltas) if d != 0]
    if nonzero:
        max_idx, max_val = max(nonzero, key=lambda t: t[1])
        min_idx, min_val = min(nonzero, key=lambda t: t[1])
        ax.annotate(f'{max_val}', xy=(x[max_idx], max_val),
                    xytext=(0, 5), textcoords='offset points', ha='center', fontsize=8)
        ax.annotate(f'{min_val}', xy=(x[min_idx], min_val),
                    xytext=(0, -12), textcoords='offset points', ha='center', fontsize=8)

plt.tight_layout()

png_path = f"{RDV}/fig_subject191_vs_109_per_chrom.png"
svg_path = f"{RDV}/fig_subject191_vs_109_per_chrom.svg"
plt.savefig(png_path, dpi=300, bbox_inches='tight')
plt.savefig(svg_path, bbox_inches='tight')
plt.close()

print(f"Regenerated {png_path}")
print(f"Regenerated {svg_path}")


In [76]:
# Manuscript figures from frozen BriTROC-1 CN validation snapshot (2026-07-25T04:32Z, n=94 profiles / 86 labeled)
import subprocess
r = subprocess.run(["python3", "/mnt/shared-workspace/shared/zeta_siphon/make_figures.py"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr[-2000:] if r.stderr else "(no stderr)")

wrote fig1_concordance_scatter
wrote fig2_paired_slopegraph
wrote fig3_auroc_forest
ALL FIGURES DONE

(no stderr)
